In [ ]:
# Jupyter cell: RawTokenDataset を作って 1 サンプルの中身を確認する

from pathlib import Path
import torch
import sys
import os
sys.path.append(os.path.abspath('..'))
from src.dataset import RawTokenDataset

# ---- 設定（必要に応じて変更）----
data_dir = Path("/root/work/data/raw/train_v1.1")   # 例: "data/train_v0" / "data/val_v0"
window_size = 6
stride = 1
filter_interrupts = True
filter_overlaps = False
idx = 0  # 見たいサンプル番号

# ---- Dataset 作成 ----
ds = RawTokenDataset(
    data_dir=data_dir,
    window_size=window_size,
    stride=stride,
    filter_interrupts=filter_interrupts,
    filter_overlaps=filter_overlaps,
)

print(f"len(ds) = {len(ds)}")

# ---- 1サンプル取得 ----
sample = ds[idx]

# ---- 中身確認 ----
print("keys:", list(sample.keys()))
for k, v in sample.items():
    print(f"{k}: dtype={v.dtype}, shape={tuple(v.shape)}, device={v.device}")

x = sample["input_ids"]
print("\n--- input_ids preview ---")
print("first 32 tokens:", x[:32].tolist())
print("last  32 tokens:", x[-32:].tolist())
print("min/max:", int(x.min().item()), int(x.max().item()))
print("num unique (approx):", int(torch.unique(x).numel()))

# labels が input_ids と同一か確認
same = torch.equal(sample["input_ids"], sample["labels"])
print("\nlabels == input_ids ?", same)

# attention_mask が全部1か確認
am = sample["attention_mask"]
print("attention_mask all ones ?", bool((am == 1).all().item()))


len(ds) = 10749324
keys: ['input_ids', 'labels', 'attention_mask']
input_ids: dtype=torch.int64, shape=(1536,), device=cpu
labels: dtype=torch.int64, shape=(1536,), device=cpu
attention_mask: dtype=torch.int64, shape=(1536,), device=cpu

--- input_ids preview ---
first 32 tokens: [15301, 10074, 77639, 12267, 76743, 208719, 146083, 4039, 69481, 30096, 30140, 32568, 21945, 126393, 29145, 21094, 9178, 174695, 69426, 7458, 14633, 57958, 3943, 70763, 233440, 36322, 103522, 43427, 121679, 1915, 11203, 132681]
last  32 tokens: [15093, 81297, 33726, 52509, 107670, 65886, 256905, 201654, 107705, 222520, 46012, 35320, 91069, 5577, 174753, 92790, 228598, 159334, 222839, 74725, 87265, 172000, 13275, 7162, 14297, 144347, 140737, 239173, 187953, 64555, 10002, 20196]
min/max: 438 261389
num unique (approx): 776

labels == input_ids ? True
attention_mask all ones ? True


In [8]:
x.shape

torch.Size([2048])

In [ ]:
from torch.utils.data import Dataset as TorchDataset
import json
import os
import re
import numpy as np
import torch
from pathlib import Path
from bisect import bisect_right


class ShardedRawTokenDataset(TorchDataset):
    """
    train_v2.0 のように、
      - metadata/metadata_{i}.json
      - segment_indices/videos/videos_{i}.bin
      - segment_indices/segment_idx_{i}.bin
    に shard 分割されているデータを読む Dataset。

    1サンプルは window_size フレーム（stride 間隔）を取り出し、
    (window_size, 32, 32) を flatten して 1D の input_ids にします。
    """

    def __init__(
        self,
        root_dir,
        window_size,
        stride=1,
        filter_interrupts=True,
        filter_overlaps=False,
        s=32,  # 32x32 が 1 frame
        token_dtype_default="uint32",
        metadata_glob="metadata_*.json",
        videos_glob="videos_*.bin",
        segment_glob="segment_idx_*.bin",
    ):
        self.root_dir = Path(root_dir)
        self.window_size = int(window_size)
        self.stride = int(stride)
        self.filter_interrupts = bool(filter_interrupts)
        self.filter_overlaps = bool(filter_overlaps)
        self.s = int(s)

        # フレーム間隔を考慮した「最初と最後のフレームの距離」
        self.video_len = (self.window_size - 1) * self.stride

        meta_dir = self.root_dir / "metadata"
        vid_dir = self.root_dir / "segment_indices" / "videos"
        seg_dir = self.root_dir / "segment_indices"

        if not meta_dir.is_dir():
            raise FileNotFoundError(f"metadata dir not found: {meta_dir}")
        if not vid_dir.is_dir():
            raise FileNotFoundError(f"videos dir not found: {vid_dir}")
        if not seg_dir.is_dir():
            raise FileNotFoundError(f"segment_indices dir not found: {seg_dir}")

        # shard index をファイル名から抽出するための正規表現
        def _extract_idx(p: Path):
            m = re.search(r"(\d+)", p.stem)
            if m is None:
                raise ValueError(f"Cannot parse shard index from: {p.name}")
            return int(m.group(1))

        # metadata_{i}.json を列挙
        meta_files = sorted(meta_dir.glob(metadata_glob), key=_extract_idx)
        if len(meta_files) == 0:
            raise FileNotFoundError(f"No metadata files found in {meta_dir} with {metadata_glob}")

        # shard 情報を保持
        self.shards = []  # list of dict: {idx, meta, video_path, segment_path, token_dtype, num_images, ...}

        for mf in meta_files:
            shard_idx = _extract_idx(mf)
            video_path = vid_dir / f"videos_{shard_idx}.bin"
            segment_path = seg_dir / f"segment_idx_{shard_idx}.bin"

            if not video_path.is_file():
                raise FileNotFoundError(f"video bin not found: {video_path}")
            if self.filter_interrupts and not segment_path.is_file():
                raise FileNotFoundError(
                    f"segment idx bin not found (required for filter_interrupts=True): {segment_path}"
                )

            meta = json.loads(mf.read_text())

            # num_images のキーが違う可能性に備えてフォールバック
            if "num_images" in meta:
                num_images = int(meta["num_images"])
            elif "n_frames" in meta:
                num_images = int(meta["n_frames"])
            elif "length" in meta:
                num_images = int(meta["length"])
            else:
                raise KeyError(f"metadata_{shard_idx}.json: cannot find num_images-like key")

            # token dtype は metadata にあれば優先
            token_dtype = np.dtype(meta.get("token_dtype", token_dtype_default))

            self.shards.append(
                dict(
                    shard_idx=shard_idx,
                    meta=meta,
                    num_images=num_images,
                    token_dtype=token_dtype,
                    video_path=video_path,
                    segment_path=segment_path if segment_path.is_file() else None,
                )
            )

        # memmap は __getitem__ で初回アクセス時に作る（lazy）
        self._video_mm = [None] * len(self.shards)
        self._seg_mm = [None] * len(self.shards)

        # 各 shard ごとに valid_start_inds を作る
        self.valid_start_inds_by_shard = []
        self.cum_counts = [0]  # prefix sum for dataset indexing

        for si, sh in enumerate(self.shards):
            num_images = sh["num_images"]

            if num_images <= self.video_len:
                valid = []
                self.valid_start_inds_by_shard.append(valid)
                self.cum_counts.append(self.cum_counts[-1] + len(valid))
                continue

            # segment_ids が必要ならここで memmap（構築時に一度だけ）
            segment_ids = None
            if self.filter_interrupts:
                segment_ids = np.memmap(
                    sh["segment_path"],
                    dtype=np.int32,
                    mode="r",
                    shape=(num_images,),
                )

            valid = []
            # start_ind は [0, num_images - video_len - 1] の範囲
            for start_ind in range(num_images - self.video_len):
                if self.filter_interrupts:
                    if segment_ids[start_ind] != segment_ids[start_ind + self.video_len]:
                        continue
                valid.append(start_ind)

            if self.filter_overlaps:
                filtered = []
                for start_ind in valid:
                    overlapping = {start_ind - i * self.stride for i in range(1, self.window_size)}
                    for existing in filtered[-self.window_size * self.stride:]:
                        if existing in overlapping:
                            break
                    else:
                        filtered.append(start_ind)
                valid = filtered

            self.valid_start_inds_by_shard.append(valid)
            self.cum_counts.append(self.cum_counts[-1] + len(valid))

        self._length = self.cum_counts[-1]

        if self._length == 0:
            raise RuntimeError(
                "No valid sequences found. Check window_size/stride/filter_interrupts and your segment_idx files."
            )

    def __len__(self):
        return self._length

    def _ensure_memmaps(self, shard_i: int):
        """動画トークンと segment_ids の memmap を lazy で作る"""
        if self._video_mm[shard_i] is None:
            sh = self.shards[shard_i]
            shape = (sh["num_images"], self.s, self.s)
            self._video_mm[shard_i] = np.memmap(
                sh["video_path"],
                dtype=sh["token_dtype"],
                mode="r",
                shape=shape,
            )
        if self.filter_interrupts and self._seg_mm[shard_i] is None:
            sh = self.shards[shard_i]
            self._seg_mm[shard_i] = np.memmap(
                sh["segment_path"],
                dtype=np.int32,
                mode="r",
                shape=(sh["num_images"],),
            )

    def __getitem__(self, idx: int):
        # 全体 idx -> shard を二分探索
        if idx < 0 or idx >= self._length:
            raise IndexError(idx)

        shard_i = bisect_right(self.cum_counts, idx) - 1
        inner_idx = idx - self.cum_counts[shard_i]

        start_ind = self.valid_start_inds_by_shard[shard_i][inner_idx]

        self._ensure_memmaps(shard_i)
        data = self._video_mm[shard_i]

        # (window_size, 32, 32) を取り出して flatten
        frames = data[start_ind : start_ind + self.video_len + 1 : self.stride]  # (window_size, s, s)
        x = torch.from_numpy(frames.astype(np.int64)).flatten()                  # (window_size*s*s,)

        attention_mask = torch.ones_like(x)
        return {
            "input_ids": x,
            "labels": x,
            "attention_mask": attention_mask,
            # デバッグ用にメタ情報も返したいなら以下を追加してもOK
            # "shard_idx": self.shards[shard_i]["shard_idx"],
            # "start_ind": start_ind,
        }


In [9]:
# Jupyter cell: shard の bin サイズが metadata と一致するか検証する（train_v2.0 用）
from pathlib import Path
import json
import numpy as np

root = Path("../data/raw/train_v2.0")  # ←適宜変更

meta_dir = root / "metadata"
vid_dir  = root / "segment_indices" / "videos"
seg_dir  = root / "segment_indices"

token_dtype = np.dtype("int32")  # train_v2.0 は metadata に dtype が無い前提
s = 32                            # 32x32 tokens per frame

meta_files = sorted(meta_dir.glob("metadata_*.json"))
print("num metadata files:", len(meta_files))

def fmt_mb(nbytes: int) -> str:
    return f"{nbytes/1024/1024:.2f} MB"

bad = []
for mf in meta_files:
    meta = json.loads(mf.read_text())
    shard_ind = int(meta["shard_ind"])
    nframes   = int(meta["shard_num_frames"])

    video_path = vid_dir / f"video_{shard_ind}.bin"
    seg_path   = seg_dir / f"segment_idx_{shard_ind}.bin"

    # --- expected sizes ---
    expected_video_bytes = nframes * s * s * token_dtype.itemsize
    expected_seg_bytes   = nframes * np.dtype("int32").itemsize

    # --- actual sizes ---
    if not video_path.is_file():
        bad.append((shard_ind, "missing video"))
        print(f"[{shard_ind:03d}] MISSING video: {video_path}")
        continue

    actual_video_bytes = video_path.stat().st_size

    seg_status = ""
    actual_seg_bytes = None
    if seg_path.is_file():
        actual_seg_bytes = seg_path.stat().st_size
        if actual_seg_bytes != expected_seg_bytes:
            seg_status = f"SEG_MISMATCH exp={expected_seg_bytes} act={actual_seg_bytes}"
    else:
        seg_status = "SEG_MISSING"

    ok_video = (actual_video_bytes == expected_video_bytes)

    if not ok_video or ("MISMATCH" in seg_status) or ("MISSING" in seg_status):
        bad.append((shard_ind, "size mismatch or missing"))
        print(
            f"[{shard_ind:03d}] "
            f"VIDEO exp={expected_video_bytes} ({fmt_mb(expected_video_bytes)}) "
            f"act={actual_video_bytes} ({fmt_mb(actual_video_bytes)}) "
            f"=> {'OK' if ok_video else 'MISMATCH'} | {seg_status}"
        )

print("\n---- summary ----")
print("bad shards:", len(bad))
if len(bad) > 0:
    print("first 20 bad:", bad[:20])
else:
    print("all shards look consistent ✅")


num metadata files: 100
[000] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[001] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[010] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[011] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[012] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[013] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[014] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[015] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[016] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[017] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[018] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[019] VIDEO exp=460972032 (439.62 MB) act=81358848 (77.59 MB) => MISMATCH | 
[002] VIDEO exp=460972032 (439.62 MB) act=81358848 (

In [5]:
# Jupyter cell: サイズ不一致の原因を推定する診断（dtype/shape/num_frames 推定）
from pathlib import Path
import json
import numpy as np

root = Path("../data/raw/train_v2.0")  # ←適宜変更
meta_dir = root / "metadata"
vid_dir  = root / "segment_indices" / "videos"
seg_dir  = root / "segment_indices"

s = 32  # あなたの前提：1フレーム 32x32 tokens

def fmt_mb(n): 
    return f"{n/1024/1024:.2f} MB"

# まず 1 shard だけ（shard_ind=0）で見る。必要なら shard_ind を変えてください
meta = json.loads((meta_dir / "metadata_0.json").read_text())
shard_ind = int(meta["shard_ind"])
nframes_meta = int(meta["shard_num_frames"])

video_path = vid_dir / f"video_{shard_ind}.bin"
seg_path   = seg_dir / f"segment_idx_{shard_ind}.bin"

print("metadata:", meta)
print("video_path:", video_path, "exists?", video_path.is_file())
print("seg_path:", seg_path, "exists?", seg_path.is_file())

video_bytes = video_path.stat().st_size if video_path.is_file() else None
seg_bytes   = seg_path.stat().st_size if seg_path.is_file() else None

print("\n--- file sizes ---")
print("video bytes:", video_bytes, f"({fmt_mb(video_bytes)})" if video_bytes is not None else "")
print("seg   bytes:", seg_bytes,   f"({fmt_mb(seg_bytes)})"   if seg_bytes is not None else "")

print("\n--- implied num_frames from video file under candidate dtypes ---")
cand_video_dtypes = ["uint8", "uint16", "uint32", "int32"]
for dt in cand_video_dtypes:
    item = np.dtype(dt).itemsize
    denom = s * s * item
    if video_bytes % denom == 0:
        n = video_bytes // denom
        print(f"video dtype={dt:6s} -> implied frames = {n:,}  (exact)")
    else:
        n = video_bytes / denom
        print(f"video dtype={dt:6s} -> implied frames ≈ {n:,.2f}")

print("\n--- implied segment_idx length under candidate dtypes ---")
if seg_bytes is not None:
    cand_seg_dtypes = ["uint8", "uint16", "uint32", "int32", "int64"]
    for dt in cand_seg_dtypes:
        item = np.dtype(dt).itemsize
        if seg_bytes % item == 0:
            n = seg_bytes // item
            print(f"seg dtype={dt:6s} -> implied length = {n:,}")
else:
    print("seg file missing")

print("\n--- metadata frame count ---")
print("metadata shard_num_frames:", f"{nframes_meta:,}")

# ついでに、video uint32, s=32 と仮定したときの implied frames を明示
if video_bytes is not None:
    implied = video_bytes // (s*s*np.dtype("uint32").itemsize)
    rem = video_bytes % (s*s*np.dtype("uint32").itemsize)
    print("\nvideo implied frames (assuming uint32, s=32):", f"{implied:,}", " remainder:", rem)


metadata: {'shard_num_frames': 112542, 'shard_ind': 0}
video_path: ../data/raw/train_v2.0/segment_indices/videos/video_0.bin exists? True
seg_path: ../data/raw/train_v2.0/segment_indices/segment_idx_0.bin exists? True

--- file sizes ---
video bytes: 81358848 (77.59 MB)
seg   bytes: 450168 (0.43 MB)

--- implied num_frames from video file under candidate dtypes ---
video dtype=uint8  -> implied frames = 79,452  (exact)
video dtype=uint16 -> implied frames = 39,726  (exact)
video dtype=uint32 -> implied frames = 19,863  (exact)
video dtype=int32  -> implied frames = 19,863  (exact)

--- implied segment_idx length under candidate dtypes ---
seg dtype=uint8  -> implied length = 450,168
seg dtype=uint16 -> implied length = 225,084
seg dtype=uint32 -> implied length = 112,542
seg dtype=int32  -> implied length = 112,542
seg dtype=int64  -> implied length = 56,271

--- metadata frame count ---
metadata shard_num_frames: 112,542

video implied frames (assuming uint32, s=32): 19,863  remainder

In [6]:
# Jupyter cell: segment_idx_0.bin の値域・単調性・ユニーク数などを確認
from pathlib import Path
import json
import numpy as np

root = Path("../data/raw/train_v2.0")  # 適宜変更

meta = json.loads((root/"metadata"/"metadata_0.json").read_text())
shard_ind = int(meta["shard_ind"])
n_meta = int(meta["shard_num_frames"])

seg_path = root/"segment_indices"/f"segment_idx_{shard_ind}.bin"
vid_path = root/"segment_indices"/"videos"/f"video_{shard_ind}.bin"

s = 32
video_bytes = vid_path.stat().st_size
video_frames_u32 = video_bytes // (s*s*np.dtype("uint32").itemsize)

seg = np.memmap(seg_path, dtype=np.int32, mode="r", shape=(n_meta,))

print("metadata frames:", n_meta)
print("video implied frames (32x32, uint32):", video_frames_u32)
print()

print("segment_idx stats")
print("  dtype:", seg.dtype, "len:", seg.shape[0])
print("  min/max:", int(seg.min()), int(seg.max()))
print("  unique:", int(np.unique(seg).size))
print("  first 20:", seg[:20].tolist())
print("  last  20:", seg[-20:].tolist())

# 単調性チェック（RawTokenDataset の segment_ids 的に使えるかどうか）
diff = np.diff(seg.astype(np.int64))
print("\nmonotonic non-decreasing?", bool((diff >= 0).all()))
print("num decreases:", int((diff < 0).sum()))


metadata frames: 112542
video implied frames (32x32, uint32): 19863

segment_idx stats
  dtype: int32 len: 112542
  min/max: 0 520
  unique: 521
  first 20: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  last  20: [520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520]

monotonic non-decreasing? True
num decreases: 0


In [7]:
# Jupyter cell: video_0.bin の dtype を推定（値域・ユニーク・分布ざっくり）
from pathlib import Path
import numpy as np

root = Path("../data/raw/train_v2.0")  # 適宜変更
video_path = root/"segment_indices"/"videos"/"video_0.bin"

s = 32
file_bytes = video_path.stat().st_size
print("video bytes:", file_bytes)

cand = ["uint8", "uint16", "uint32"]
n_check_frames = 256  # 先頭だけ見る（必要なら増やしてOK）

for dt in cand:
    dtype = np.dtype(dt)
    denom = s*s*dtype.itemsize
    if file_bytes % denom != 0:
        print(f"\n[{dt}] not divisible -> skip")
        continue

    nframes = file_bytes // denom
    frames_to_read = min(nframes, n_check_frames)

    mm = np.memmap(video_path, dtype=dtype, mode="r", shape=(nframes, s, s))
    x = np.array(mm[:frames_to_read]).reshape(-1)  # サンプル部分だけ実配列化

    print(f"\n[{dt}] implied frames = {nframes:,}  (reading first {frames_to_read} frames)")
    print("  min/max:", int(x.min()), int(x.max()))
    print("  unique:", int(np.unique(x).size))
    # “それっぽさ”の雑な目安（疎すぎないか）
    top = np.bincount(x.astype(np.int64), minlength=min(int(x.max())+1, 100000)) if x.max() < 100000 else None
    if top is not None:
        # 最頻値トップ5
        top5 = np.argsort(top)[-5:][::-1]
        print("  top5 values:", [(int(v), int(top[v])) for v in top5])


video bytes: 81358848

[uint8] implied frames = 79,452  (reading first 256 frames)
  min/max: 0 255
  unique: 256
  top5 values: [(0, 131187), (74, 1393), (84, 1281), (132, 1189), (122, 1156)]

[uint16] implied frames = 39,726  (reading first 256 frames)
  min/max: 0 63978
  unique: 25804
  top5 values: [(0, 131072), (22520, 335), (9720, 241), (19038, 86), (19960, 76)]

[uint32] implied frames = 19,863  (reading first 256 frames)
  min/max: 39 63996
  unique: 36381
  top5 values: [(22520, 650), (9720, 404), (18565, 160), (18510, 160), (18566, 130)]


In [11]:
from pathlib import Path
import re

root = Path("../data/raw/train_v2.0")  # 適宜変更

meta_dir = root / "metadata"
seg_dir  = root / "segment_indices"
vid_dir  = seg_dir / "videos"

print("root:", root.resolve())
print("metadata dir:", meta_dir, "exists?", meta_dir.is_dir())
print("segment dir :", seg_dir, "exists?", seg_dir.is_dir())
print("videos dir  :", vid_dir, "exists?", vid_dir.is_dir())

def shard_id_from_name(name: str):
    m = re.search(r"(\d+)", name)
    return int(m.group(1)) if m else None

meta_files = sorted(meta_dir.glob("metadata_*.json"))
seg_files  = sorted(seg_dir.glob("segment_idx_*.bin"))
vid_files  = sorted(list(vid_dir.glob("video_*.bin")) + list(vid_dir.glob("videos_*.bin")))

meta_ids = sorted({shard_id_from_name(p.name) for p in meta_files if shard_id_from_name(p.name) is not None})
seg_ids  = sorted({shard_id_from_name(p.name) for p in seg_files  if shard_id_from_name(p.name) is not None})
vid_ids  = sorted({shard_id_from_name(p.name) for p in vid_files  if shard_id_from_name(p.name) is not None})

print("\ncounts")
print(" metadata:", len(meta_files), "unique ids:", len(meta_ids))
print(" segment :", len(seg_files),  "unique ids:", len(seg_ids))
print(" videos  :", len(vid_files),  "unique ids:", len(vid_ids))

print("\nid range (first/last)")
if meta_ids: print(" meta:", meta_ids[0], meta_ids[-1])
if seg_ids:  print(" seg :", seg_ids[0],  seg_ids[-1])
if vid_ids:  print(" vid :", vid_ids[0],  vid_ids[-1])

# 欠けを確認
missing_vid = sorted(set(meta_ids) - set(vid_ids))
missing_seg = sorted(set(meta_ids) - set(seg_ids))
extra_vid   = sorted(set(vid_ids)  - set(meta_ids))
extra_seg   = sorted(set(seg_ids)  - set(meta_ids))

print("\nmissing compared to metadata ids")
print(" missing video ids:", missing_vid[:20], ("..." if len(missing_vid) > 20 else ""))
print(" missing seg  ids:", missing_seg[:20], ("..." if len(missing_seg) > 20 else ""))

print("\nextra compared to metadata ids")
print(" extra video ids:", extra_vid[:20], ("..." if len(extra_vid) > 20 else ""))
print(" extra seg  ids:", extra_seg[:20], ("..." if len(extra_seg) > 20 else ""))


root: /root/work/data/raw/train_v2.0
metadata dir: ../data/raw/train_v2.0/metadata exists? True
segment dir : ../data/raw/train_v2.0/segment_indices exists? True
videos dir  : ../data/raw/train_v2.0/segment_indices/videos exists? True

counts
 metadata: 100 unique ids: 100
 segment : 100 unique ids: 100
 videos  : 100 unique ids: 100

id range (first/last)
 meta: 0 99
 seg : 0 99
 vid : 0 99

missing compared to metadata ids
 missing video ids: [] 
 missing seg  ids: [] 

extra compared to metadata ids
 extra video ids: [] 
 extra seg  ids: [] 


In [12]:
from pathlib import Path
import json
import numpy as np

root = Path("../data/raw/train_v2.0")  # 適宜変更
s = 32  # 32x32 tokens per frame

def inspect_shard(i: int):
    meta = json.loads((root/"metadata"/f"metadata_{i}.json").read_text())
    shard_ind = int(meta["shard_ind"])
    n_raw = int(meta["shard_num_frames"])

    # video ファイル名ゆれ対応
    vid_dir = root/"segment_indices"/"videos"
    candidates = [vid_dir/f"video_{shard_ind}.bin", vid_dir/f"videos_{shard_ind}.bin"]
    video_path = next((p for p in candidates if p.is_file()), None)
    seg_path = root/"segment_indices"/f"segment_idx_{shard_ind}.bin"

    print("---- shard", i, "----")
    print("meta:", meta)
    print("video:", video_path)
    print("seg  :", seg_path)

    if video_path is None or not seg_path.is_file():
        print("missing files")
        return

    vb = video_path.stat().st_size
    sb = seg_path.stat().st_size
    print("video bytes:", vb, f"({vb/1024/1024:.2f} MB)")
    print("seg   bytes:", sb, f"({sb/1024/1024:.2f} MB)")

    # segment_idx は int32 で長さ n_raw と一致するはず
    seg = np.memmap(seg_path, dtype=np.int32, mode="r", shape=(n_raw,))
    print("segment_idx: len", seg.shape[0], "min/max", int(seg.min()), int(seg.max()),
          "unique", int(np.unique(seg).size), "monotonic", bool((np.diff(seg.astype(np.int64))>=0).all()))

    # video 側 dtype 候補で “フレーム数”推定
    for dt in ["uint8","uint16","uint32"]:
        item = np.dtype(dt).itemsize
        denom = s*s*item
        if vb % denom == 0:
            n_video = vb // denom
            print(f"video dtype={dt:5s}: implied frames = {n_video:,} (exact)")
        else:
            print(f"video dtype={dt:5s}: not divisible")

# 例：shard0
inspect_shard(0)


---- shard 0 ----
meta: {'shard_num_frames': 112542, 'shard_ind': 0}
video: ../data/raw/train_v2.0/segment_indices/videos/video_0.bin
seg  : ../data/raw/train_v2.0/segment_indices/segment_idx_0.bin
video bytes: 81358848 (77.59 MB)
seg   bytes: 450168 (0.43 MB)
segment_idx: len 112542 min/max 0 520 unique 521 monotonic True
video dtype=uint8: implied frames = 79,452 (exact)
video dtype=uint16: implied frames = 39,726 (exact)
video dtype=uint32: implied frames = 19,863 (exact)


In [15]:
from pathlib import Path
import json
import numpy as np

root = Path("../data/raw/train_v2.0")
s = 32
video_dtype = np.int32  # 今のログだと最有力。あとで変えてOK

meta_files = sorted((root/"metadata").glob("metadata_*.json"))

rows = []
for mf in meta_files[:20]:  # まずは先頭20 shardだけ。増やしたければ数字を変更
    meta = json.loads(mf.read_text())
    i = int(meta["shard_ind"])
    n_raw = int(meta["shard_num_frames"])

    vid_dir = root/"segment_indices"/"videos"
    video_path = (vid_dir/f"video_{i}.bin")
    if not video_path.is_file():
        video_path = (vid_dir/f"videos_{i}.bin")
    seg_path = root/"segment_indices"/f"segment_idx_{i}.bin"

    if not video_path.is_file() or not seg_path.is_file():
        continue

    vb = video_path.stat().st_size
    denom = s*s*np.dtype(video_dtype).itemsize
    n_video = vb // denom if vb % denom == 0 else None

    rows.append((i, n_raw, n_video, (n_raw / n_video if n_video else None)))

print("shard_ind | raw_frames | video_frames(u16) | raw/video")
for r in rows:
    print(f"{r[0]:8d} | {r[1]:9,d} | {r[2]:15,d} | {r[3]:.3f}" if r[2] else f"{r[0]:8d} | {r[1]:9,d} | {'?':>15} | ?")


shard_ind | raw_frames | video_frames(u16) | raw/video
       0 |   112,542 |          19,863 | 5.666
       1 |   112,542 |          19,863 | 5.666
      10 |   112,542 |          19,863 | 5.666
      11 |   112,542 |          19,863 | 5.666
      12 |   112,542 |          19,863 | 5.666
      13 |   112,542 |          19,863 | 5.666
      14 |   112,542 |          19,863 | 5.666
      15 |   112,542 |          19,863 | 5.666
      16 |   112,542 |          19,863 | 5.666
      17 |   112,542 |          19,863 | 5.666
      18 |   112,542 |          19,863 | 5.666
      19 |   112,542 |          19,863 | 5.666
       2 |   112,542 |          19,863 | 5.666
      20 |   112,542 |          19,863 | 5.666
      21 |   112,542 |          19,863 | 5.666
      22 |   112,542 |          19,863 | 5.666
      23 |   112,542 |          19,863 | 5.666
      24 |   112,542 |          19,863 | 5.666
      25 |   112,542 |          19,863 | 5.666
      26 |   112,542 |          19,863 | 5.666


In [14]:
# Jupyter cell: video_0.bin を int32 としても統計確認（uint8/uint16/uint32/int32 を比較）
from pathlib import Path
import numpy as np

root = Path("../data/raw/train_v2.0")  # 適宜変更
video_path = root/"segment_indices"/"videos"/"video_0.bin"  # 必要なら videos_0.bin に変更

s = 32
file_bytes = video_path.stat().st_size
print("video bytes:", file_bytes)

cand = ["uint8", "uint16", "uint32", "int32"]
n_check_frames = 256  # 先頭だけ見る

for dt in cand:
    dtype = np.dtype(dt)
    denom = s*s*dtype.itemsize
    if file_bytes % denom != 0:
        print(f"\n[{dt}] not divisible -> skip")
        continue

    nframes = file_bytes // denom
    frames_to_read = min(nframes, n_check_frames)

    mm = np.memmap(video_path, dtype=dtype, mode="r", shape=(nframes, s, s))
    x = np.array(mm[:frames_to_read]).reshape(-1)  # サンプル部分だけ実配列化

    print(f"\n[{dt}] implied frames = {nframes:,}  (reading first {frames_to_read} frames)")
    print("  min/max:", int(x.min()), int(x.max()))
    print("  unique:", int(np.unique(x).size))

    # 最頻値トップ5（値域が小さめのときだけ）
    if x.max() < 100000 and x.min() >= 0:
        bc = np.bincount(x.astype(np.int64), minlength=int(x.max())+1)
        top5 = np.argsort(bc)[-5:][::-1]
        print("  top5 values:", [(int(v), int(bc[v])) for v in top5])
    else:
        # 値域が広い/負値がある場合は bincount を避ける
        vals, counts = np.unique(x, return_counts=True)
        order = np.argsort(counts)[-5:][::-1]
        print("  top5 values:", [(int(vals[j]), int(counts[j])) for j in order])


video bytes: 81358848

[uint8] implied frames = 79,452  (reading first 256 frames)
  min/max: 0 255
  unique: 256
  top5 values: [(0, 131187), (74, 1393), (84, 1281), (132, 1189), (122, 1156)]

[uint16] implied frames = 39,726  (reading first 256 frames)
  min/max: 0 63978
  unique: 25804
  top5 values: [(0, 131072), (22520, 335), (9720, 241), (19038, 86), (19960, 76)]

[uint32] implied frames = 19,863  (reading first 256 frames)
  min/max: 39 63996
  unique: 36381
  top5 values: [(22520, 650), (9720, 404), (18565, 160), (18510, 160), (18566, 130)]

[int32] implied frames = 19,863  (reading first 256 frames)
  min/max: 39 63996
  unique: 36381
  top5 values: [(22520, 650), (9720, 404), (18565, 160), (18510, 160), (18566, 130)]


In [17]:
from pathlib import Path
import json
import numpy as np

root = Path("../data/raw/train_v2.0").resolve()  # ★絶対パスに揃える
rank = 0

# ---- load metadata (root) ----
meta_root = json.loads((root/"metadata.json").read_text())
print("root:", root)
print("root metadata:", meta_root)

# ---- load shard metadata（両対応）----
cand = [
    root / f"metadata_{rank}.json",
    root / "metadata" / f"metadata_{rank}.json",
]
shard_meta_path = next((p for p in cand if p.is_file()), None)
assert shard_meta_path is not None, f"metadata_{rank}.json not found in {cand}"

meta_shard = json.loads(shard_meta_path.read_text())
total_frames = int(meta_shard["shard_num_frames"])
shard_ind = int(meta_shard["shard_ind"])
print("\nshard meta path:", shard_meta_path)
print("shard meta:", meta_shard)
print("total_frames:", total_frames, "hz:", meta_root.get("hz"))

# ---- find all *_{rank}.bin under root (and common subdirs) ----
search_dirs = [
    root,
    root/"segment_indices",
    root/"segment_indices"/"videos",
    root/"metadata",
]
files = []
for d in search_dirs:
    if d.is_dir():
        files += list(d.glob(f"*_{shard_ind}.bin"))

# 重複除去（絶対パスで統一）
files = sorted({p.resolve() for p in files})

print("\nfound files:")
for p in files:
    # p は絶対パス、root も絶対パスなので relative_to が安全
    print("-", p.relative_to(root), "bytes:", p.stat().st_size)

# ---- helper: infer per-frame payload size for candidate dtypes ----
cand_dtypes = [np.uint8, np.uint16, np.int32, np.uint32, np.float16, np.float32, np.float64]

print("\n---- infer 'elements per frame' by dtype (file_size / total_frames / itemsize) ----")
for p in files:
    size = p.stat().st_size
    print(f"\n{p.relative_to(root)}  size={size}")
    for dt in cand_dtypes:
        item = np.dtype(dt).itemsize
        denom = total_frames * item
        if denom != 0 and size % denom == 0:
            elems = size // denom
            print(f"  dtype={np.dtype(dt).name:7s} -> elems/frame = {elems}")


root: /root/work/data/raw/train_v2.0
root metadata: {'num_shards': 100, 'query': None, 'hz': 30, 'num_images': 11254161}

shard meta path: /root/work/data/raw/train_v2.0/metadata/metadata_0.json
shard meta: {'shard_num_frames': 112542, 'shard_ind': 0}
total_frames: 112542 hz: 30

found files:
- segment_indices/segment_idx_0.bin bytes: 450168
- segment_indices/videos/video_0.bin bytes: 81358848

---- infer 'elements per frame' by dtype (file_size / total_frames / itemsize) ----

segment_indices/segment_idx_0.bin  size=450168
  dtype=uint8   -> elems/frame = 4
  dtype=uint16  -> elems/frame = 2
  dtype=int32   -> elems/frame = 1
  dtype=uint32  -> elems/frame = 1
  dtype=float16 -> elems/frame = 2
  dtype=float32 -> elems/frame = 1

segment_indices/videos/video_0.bin  size=81358848


In [18]:
from pathlib import Path
import json
import numpy as np

root = Path("/root/work/data/raw/train_v2.0")
shard = 0

meta = json.loads((root/"metadata"/f"metadata_{shard}.json").read_text())
i = int(meta["shard_ind"])
n_raw = int(meta["shard_num_frames"])

# raw: segment ids (30Hz)
seg_raw = np.memmap(root/"segment_indices"/f"segment_idx_{i}.bin",
                    dtype=np.int32, mode="r", shape=(n_raw,))

# raw: clip boundaries
change = np.where(np.diff(seg_raw) != 0)[0] + 1
raw_bounds = np.concatenate([[0], change, [n_raw]])
raw_lens = np.diff(raw_bounds)

print("raw clips:", len(raw_lens))
print("raw clip length stats: min/mean/max =", int(raw_lens.min()), float(raw_lens.mean()), int(raw_lens.max()))
print("raw seg id range:", int(seg_raw.min()), int(seg_raw.max()))

# video: token timeline length (NOT raw)
video_path = root/"segment_indices"/"videos"/f"video_{i}.bin"
vb = video_path.stat().st_size
s = 32

# dtypeはログからまず uint16 が最有力
video_dtype = np.uint16
n_video = vb // (s*s*np.dtype(video_dtype).itemsize)
print("\nvideo token frames:", n_video, "dtype:", video_dtype)

# raw index -> video index の線形写像（境界投影）
# video_index = floor(raw_index * n_video / n_raw)
vid_bounds = (raw_bounds.astype(np.int64) * n_video) // n_raw
# 0..n_video にクリップ境界が入る
vid_bounds[0] = 0
vid_bounds[-1] = n_video
vid_lens = np.diff(vid_bounds)

print("projected video clips:", len(vid_lens))
print("video clip length stats: min/mean/max =", int(vid_lens.min()), float(vid_lens.mean()), int(vid_lens.max()))
print("ratio raw/video mean:", float(raw_lens.mean()/max(vid_lens.mean(),1)))

# クリップ長が離散的（固定長に近い）かを見るために上位頻度
vals, cnt = np.unique(vid_lens, return_counts=True)
order = np.argsort(cnt)[-10:][::-1]
print("\nmost common video-clip lengths (top10):")
for j in order:
    print(f"  len={int(vals[j])}: count={int(cnt[j])}")

print("\nfirst 15 clips raw_len -> vid_len:")
for k in range(15):
    print(f"  clip{k:02d}: raw={int(raw_lens[k])} -> vid={int(vid_lens[k])}")


raw clips: 521
raw clip length stats: min/mean/max = 1 216.01151631477927 1882
raw seg id range: 0 520

video token frames: 39726 dtype: <class 'numpy.uint16'>
projected video clips: 521
video clip length stats: min/mean/max = 0 76.24952015355086 664
ratio raw/video mean: 2.8329557468660322

most common video-clip lengths (top10):
  len=0: count=117
  len=1: count=77
  len=125: count=9
  len=115: count=8
  len=2: count=8
  len=130: count=7
  len=134: count=6
  len=144: count=6
  len=117: count=6
  len=118: count=6

first 15 clips raw_len -> vid_len:
  clip00: raw=806 -> vid=284
  clip01: raw=970 -> vid=342
  clip02: raw=1103 -> vid=390
  clip03: raw=837 -> vid=295
  clip04: raw=1006 -> vid=355
  clip05: raw=1176 -> vid=415
  clip06: raw=1027 -> vid=363
  clip07: raw=1026 -> vid=362
  clip08: raw=188 -> vid=66
  clip09: raw=1 -> vid=1
  clip10: raw=320 -> vid=113
  clip11: raw=1 -> vid=0
  clip12: raw=284 -> vid=100
  clip13: raw=1 -> vid=1
  clip14: raw=291 -> vid=102


In [19]:
import numpy as np
from pathlib import Path

root = Path("/root/work/data/raw/train_v2.0")
video_path = root/"segment_indices"/"videos"/"video_0.bin"
s = 32
vb = video_path.stat().st_size

n32 = vb // (s*s*np.dtype(np.int32).itemsize)
mm32 = np.memmap(video_path, dtype=np.int32, mode="r", shape=(n32, s, s))

# int32の下位16bit/上位16bitを取り出す
x32 = np.array(mm32[0]).reshape(-1).astype(np.uint32)
low  = (x32 & 0xFFFF).astype(np.uint16)
high = ((x32 >> 16) & 0xFFFF).astype(np.uint16)

print("int32 frame0 stats:", int(mm32[0].min()), int(mm32[0].max()))
print("low  stats:", int(low.min()), int(low.max()), "zeros:", int((low==0).sum()))
print("high stats:", int(high.min()), int(high.max()), "zeros:", int((high==0).sum()))
print("high all-zero?", bool((high==0).all()))


int32 frame0 stats: 124 63829
low  stats: 124 63829 zeros: 0
high stats: 0 0 zeros: 1024
high all-zero? True


In [20]:
from pathlib import Path
import json
import numpy as np

root = Path("/root/work/data/raw/train_v2.0")
shard = 0
s = 32

meta = json.loads((root/"metadata"/f"metadata_{shard}.json").read_text())
i = int(meta["shard_ind"])
n_raw = int(meta["shard_num_frames"])

# raw segment ids
seg_raw = np.memmap(root/"segment_indices"/f"segment_idx_{i}.bin",
                    dtype=np.int32, mode="r", shape=(n_raw,))

# token video (int32)
video_path = root/"segment_indices"/"videos"/f"video_{i}.bin"
vb = video_path.stat().st_size
n_tok = vb // (s*s*np.dtype(np.int32).itemsize)

print("n_raw:", n_raw, "n_tok:", n_tok, "raw/tok:", n_raw/n_tok)

# token frame -> raw frame index (linear map)
raw_index = (np.arange(n_tok, dtype=np.int64) * n_raw) // n_tok
seg_tok = np.array(seg_raw[raw_index], dtype=np.int32)

print("seg_tok range:", int(seg_tok.min()), int(seg_tok.max()), "unique:", int(np.unique(seg_tok).size))
print("monotonic(seg_tok)?", bool((np.diff(seg_tok.astype(np.int64)) >= 0).all()))

# token 側で segment 長を見る（0長問題が出ない）
change = np.where(np.diff(seg_tok) != 0)[0] + 1
bounds = np.concatenate([[0], change, [n_tok]])
tok_lens = np.diff(bounds)

print("token segments:", len(tok_lens))
print("token seg length stats: min/mean/max =", int(tok_lens.min()), float(tok_lens.mean()), int(tok_lens.max()))
vals, cnt = np.unique(tok_lens, return_counts=True)
order = np.argsort(cnt)[-10:][::-1]
print("\nmost common token-seg lengths (top10):")
for j in order:
    print(f"  len={int(vals[j])}: count={int(cnt[j])}")


n_raw: 112542 n_tok: 19863 raw/tok: 5.6659114937320645
seg_tok range: 0 520 unique: 369
monotonic(seg_tok)? True
token segments: 369
token seg length stats: min/mean/max = 1 53.829268292682926 332

most common token-seg lengths (top10):
  len=1: count=51
  len=63: count=16
  len=58: count=12
  len=59: count=11
  len=65: count=11
  len=67: count=10
  len=66: count=9
  len=72: count=8
  len=62: count=7
  len=68: count=7


In [21]:
import numpy as np
import json
from pathlib import Path

root = Path("/root/work/data/raw/train_v2.0")
shard = 0
s = 32

meta = json.loads((root/"metadata"/f"metadata_{shard}.json").read_text())
i = int(meta["shard_ind"])
n_raw = int(meta["shard_num_frames"])

# token length (int32, 32x32)
video_path = root/"segment_indices"/"videos"/f"video_{i}.bin"
vb = video_path.stat().st_size
n_tok = vb // (s*s*np.dtype(np.int32).itemsize)

# raw clip lengths from segment_idx
seg_raw = np.memmap(root/"segment_indices"/f"segment_idx_{i}.bin", dtype=np.int32, mode="r", shape=(n_raw,))
change = np.where(np.diff(seg_raw) != 0)[0] + 1
raw_bounds = np.concatenate([[0], change, [n_raw]])
clip_lens = np.diff(raw_bounds)

print("n_raw:", n_raw, "n_tok(actual):", n_tok, "raw/tok:", n_raw/n_tok)
print("num clips:", len(clip_lens), "min/mean/max clip_len:", int(clip_lens.min()), float(clip_lens.mean()), int(clip_lens.max()))

# --- 仮説: clipごとに T_k = ceil(L_k / r) を足したものが n_tok に合う ---
def sum_tokens_from_r(r: float):
    return int(np.sum(np.ceil(clip_lens / r)))

# まず r = n_raw/n_tok で試す
r0 = n_raw / n_tok
pred0 = sum_tokens_from_r(r0)
print("\ntry r0 = n_raw/n_tok =", r0)
print("pred sum ceil(L/r0) =", pred0, "diff:", pred0 - n_tok)

# r を掃引して一番合うところを探す（探索）
rs = np.linspace(r0*0.7, r0*1.3, 241)
pred = np.array([sum_tokens_from_r(r) for r in rs])
best_j = int(np.argmin(np.abs(pred - n_tok)))
print("\nbest r in sweep:", float(rs[best_j]))
print("pred:", int(pred[best_j]), "diff:", int(pred[best_j] - n_tok))

# 参考：floor の場合も
def sum_tokens_floor(r: float):
    return int(np.sum(np.floor(clip_lens / r)))

pred0f = sum_tokens_floor(r0)
print("\nfloor at r0:", pred0f, "diff:", pred0f - n_tok)


n_raw: 112542 n_tok(actual): 19863 raw/tok: 5.6659114937320645
num clips: 521 min/mean/max clip_len: 1 216.01151631477927 1882

try r0 = n_raw/n_tok = 5.6659114937320645
pred sum ceil(L/r0) = 20192 diff: 329

best r in sweep: 5.765064944872376
pred: 19845 diff: -18

floor at r0: 19671 diff: -192


In [22]:
import numpy as np
import json
from pathlib import Path

root = Path("/root/work/data/raw/train_v2.0")
shard = 0
s = 32

meta = json.loads((root/"metadata"/f"metadata_{shard}.json").read_text())
i = int(meta["shard_ind"])
n_raw = int(meta["shard_num_frames"])

video_path = root/"segment_indices"/"videos"/f"video_{i}.bin"
vb = video_path.stat().st_size
n_tok = vb // (s*s*np.dtype(np.int32).itemsize)

seg_raw = np.memmap(root/"segment_indices"/f"segment_idx_{i}.bin",
                    dtype=np.int32, mode="r", shape=(n_raw,))
change = np.where(np.diff(seg_raw) != 0)[0] + 1
raw_bounds = np.concatenate([[0], change, [n_raw]])
clip_lens = np.diff(raw_bounds)

# さっきの best r を使う（必要なら r を固定してもよい）
r = 5.765064944872376

def pred_sum(min_len: int):
    lens = clip_lens[clip_lens >= min_len]
    return int(np.sum(np.ceil(lens / r))), int(lens.size)

print("n_tok(actual):", n_tok, "using r:", r)
for th in [1,2,3,4,5,6,7,8,10,12,15,20]:
    ps, k = pred_sum(th)
    print(f"min_len>={th:2d}: pred={ps:5d} diff={ps-n_tok:+5d}  kept_clips={k:3d} dropped={len(clip_lens)-k}")


n_tok(actual): 19863 using r: 5.765064944872376
min_len>= 1: pred=19845 diff=  -18  kept_clips=521 dropped=0
min_len>= 2: pred=19655 diff= -208  kept_clips=331 dropped=190
min_len>= 3: pred=19655 diff= -208  kept_clips=331 dropped=190
min_len>= 4: pred=19652 diff= -211  kept_clips=328 dropped=193
min_len>= 5: pred=19649 diff= -214  kept_clips=325 dropped=196
min_len>= 6: pred=19648 diff= -215  kept_clips=324 dropped=197
min_len>= 7: pred=19642 diff= -221  kept_clips=321 dropped=200
min_len>= 8: pred=19638 diff= -225  kept_clips=319 dropped=202
min_len>=10: pred=19632 diff= -231  kept_clips=316 dropped=205
min_len>=12: pred=19626 diff= -237  kept_clips=313 dropped=208
min_len>=15: pred=19626 diff= -237  kept_clips=313 dropped=208
min_len>=20: pred=19616 diff= -247  kept_clips=310 dropped=211


In [23]:
import numpy as np
import json
from pathlib import Path

root = Path("/root/work/data/raw/train_v2.0")
shard = 0
s = 32

meta = json.loads((root/"metadata"/f"metadata_{shard}.json").read_text())
i = int(meta["shard_ind"])
n_raw = int(meta["shard_num_frames"])

video_path = root/"segment_indices"/"videos"/f"video_{i}.bin"
vb = video_path.stat().st_size
n_tok = vb // (s*s*np.dtype(np.int32).itemsize)

seg_raw = np.memmap(root/"segment_indices"/f"segment_idx_{i}.bin",
                    dtype=np.int32, mode="r", shape=(n_raw,))
change = np.where(np.diff(seg_raw) != 0)[0] + 1
raw_bounds = np.concatenate([[0], change, [n_raw]])  # clip start positions

# raw start -> nearest token index (round)
tok_starts = np.rint(raw_bounds.astype(np.float64) * n_tok / n_raw).astype(np.int64)
tok_starts = np.clip(tok_starts, 0, n_tok-1)

# 近傍±1 にも token が“割り当たってる”かを見るため、token index の集合を作る
tok_indices = set(range(n_tok))  # tokenは全インデックスに存在するので、ここでは簡単化
# 本当に見たいのは「境界が token グリッドにどれだけ乗るか」なので、start が同じに潰れる頻度を見る
unique_starts = len(np.unique(tok_starts))
print("num clips:", len(raw_bounds)-1)
print("unique projected token-starts:", unique_starts)
print("clips whose projected start collides (lost uniqueness):", (len(raw_bounds)-1) - unique_starts)

# startの衝突分布（どれくらい多重に潰れているか）
vals, cnt = np.unique(tok_starts, return_counts=True)
print("max collision multiplicity:", int(cnt.max()))
print("how many token positions have >=2 clip-starts mapped onto them:", int((cnt>=2).sum()))


num clips: 521
unique projected token-starts: 363
clips whose projected start collides (lost uniqueness): 158
max collision multiplicity: 2
how many token positions have >=2 clip-starts mapped onto them: 159


In [24]:
import numpy as np
import json
from pathlib import Path

root = Path("/root/work/data/raw/train_v2.0")
shard = 0
s = 32

meta = json.loads((root/"metadata"/f"metadata_{shard}.json").read_text())
i = int(meta["shard_ind"])
n_raw = int(meta["shard_num_frames"])

video_path = root/"segment_indices"/"videos"/f"video_{i}.bin"
vb = video_path.stat().st_size
n_tok = vb // (s*s*np.dtype(np.int32).itemsize)

# token index t が参照している raw index を（線形写像で）作る
raw_index = (np.arange(n_tok, dtype=np.int64) * n_raw) // n_tok
step = np.diff(raw_index)

vals, cnt = np.unique(step, return_counts=True)
order = np.argsort(cnt)[::-1]

print("n_raw/n_tok:", n_raw/n_tok)
print("raw_index step stats: min/mean/max =", int(step.min()), float(step.mean()), int(step.max()))
print("most common steps:")
for j in order[:10]:
    print(f"  step={int(vals[j])}: count={int(cnt[j])}")


n_raw/n_tok: 5.6659114937320645
raw_index step stats: min/mean/max = 5 5.665894673245393 6
most common steps:
  step=6: count=13226
  step=5: count=6636


In [25]:
import numpy as np, json
from pathlib import Path

root = Path("/root/work/data/raw/train_v2.0")

def load_seg(shard: int):
    meta = json.loads((root/"metadata"/f"metadata_{shard}.json").read_text())
    n = int(meta["shard_num_frames"])
    i = int(meta["shard_ind"])
    seg = np.memmap(root/"segment_indices"/f"segment_idx_{i}.bin", dtype=np.int32, mode="r", shape=(n,))
    return seg

seg0 = load_seg(0)
seg1 = load_seg(1)

print("shard0: len", len(seg0), "min/max", int(seg0.min()), int(seg0.max()),
      "first/last", int(seg0[0]), int(seg0[-1]))
print("shard1: len", len(seg1), "min/max", int(seg1.min()), int(seg1.max()),
      "first/last", int(seg1[0]), int(seg1[-1]))

print("\nlast20 shard0:", seg0[-20:].tolist())
print("first20 shard1:", seg1[:20].tolist())

print("\nDoes shard1 start with shard0 last id (continuation)?", bool(seg1[0] == seg0[-1]))
print("Does shard1 start with shard0 last id + 1 (global continuation)?", bool(seg1[0] == seg0[-1] + 1))


shard0: len 112542 min/max 0 520 first/last 0 520
shard1: len 112542 min/max 520 844 first/last 520 844

last20 shard0: [520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520]
first20 shard1: [520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520]

Does shard1 start with shard0 last id (continuation)? True
Does shard1 start with shard0 last id + 1 (global continuation)? False


In [26]:
import numpy as np, json
from pathlib import Path

root = Path("/root/work/data/raw/train_v2.0")

def load_seg(shard: int):
    meta = json.loads((root/"metadata"/f"metadata_{shard}.json").read_text())
    n = int(meta["shard_num_frames"])
    i = int(meta["shard_ind"])
    return np.memmap(root/"segment_indices"/f"segment_idx_{i}.bin", dtype=np.int32, mode="r", shape=(n,))

overlaps = []
gaps = []
for k in range(99):
    a = load_seg(k)
    b = load_seg(k+1)
    last_a = int(a[-1])
    first_b = int(b[0])
    overlaps.append(first_b == last_a)
    gaps.append(first_b == last_a + 1)

print("boundaries (k->k+1) where first == last:", sum(overlaps), "/ 99")
print("boundaries where first == last+1:", sum(gaps), "/ 99")

# どの境界がどうなってるか（最初のいくつか）
for k in range(5):
    a = load_seg(k); b = load_seg(k+1)
    print(f"{k}->{k+1}: last={int(a[-1])}, first={int(b[0])}, min/max next={int(b.min())}/{int(b.max())}")


boundaries (k->k+1) where first == last: 97 / 99
boundaries where first == last+1: 2 / 99
0->1: last=520, first=520, min/max next=520/844
1->2: last=844, first=844, min/max next=844/1375
2->3: last=1375, first=1375, min/max next=1375/1944
3->4: last=1944, first=1944, min/max next=1944/2394
4->5: last=2394, first=2394, min/max next=2394/2899


In [27]:
import json
from pathlib import Path
import numpy as np
from collections import defaultdict

root = Path("/root/work/data/raw/train_v2.0")
s = 32

# ---- helper: load raw segment_idx for shard k ----
def load_seg_raw(k: int):
    meta = json.loads((root/"metadata"/f"metadata_{k}.json").read_text())
    n_raw = int(meta["shard_num_frames"])
    i = int(meta["shard_ind"])
    seg = np.memmap(root/"segment_indices"/f"segment_idx_{i}.bin", dtype=np.int32, mode="r", shape=(n_raw,))
    return seg, n_raw, i

# ---- helper: token length for shard id i (video_i.bin) ----
def get_n_tok(i: int):
    video_path = root/"segment_indices"/"videos"/f"video_{i}.bin"
    vb = video_path.stat().st_size
    n_tok = vb // (s*s*np.dtype(np.int32).itemsize)
    return n_tok

# ---- helper: build seg_tok by linear mapping token->raw ----
def build_seg_tok(seg_raw: np.ndarray, n_raw: int, n_tok: int):
    # token index t -> raw index floor(t * n_raw / n_tok)
    raw_index = (np.arange(n_tok, dtype=np.int64) * n_raw) // n_tok
    return np.array(seg_raw[raw_index], dtype=np.int32)

num_shards = json.loads((root/"metadata.json").read_text())["num_shards"]
print("num_shards:", num_shards)

# --- boundary continuity stats on token side ---
same_last_first = 0
plus1_last_first = 0
other = 0
examples = []

# --- token counts per global segment id ---
token_count = defaultdict(int)

prev_seg_tok_last = None

for k in range(num_shards):
    seg_raw, n_raw, i = load_seg_raw(k)
    n_tok = get_n_tok(i)
    seg_tok = build_seg_tok(seg_raw, n_raw, n_tok)

    # count tokens per id (seg_tok は tokenフレームごとの segment id)
    vals, cnt = np.unique(seg_tok, return_counts=True)
    for v, c in zip(vals.tolist(), cnt.tolist()):
        token_count[int(v)] += int(c)

    # boundary check (k-1 -> k)
    if k > 0:
        first = int(seg_tok[0])
        last_prev = int(prev_seg_tok_last)
        if first == last_prev:
            same_last_first += 1
        elif first == last_prev + 1:
            plus1_last_first += 1
        else:
            other += 1
            if len(examples) < 10:
                examples.append((k-1, k, last_prev, first))

    prev_seg_tok_last = int(seg_tok[-1])

print("\n[token-side boundaries k->k+1]")
print(" first == last:", same_last_first, "/", num_shards-1)
print(" first == last+1:", plus1_last_first, "/", num_shards-1)
print(" other:", other, "/", num_shards-1)

if examples:
    print("\nexamples of 'other' boundaries (prev->curr: last_prev -> first_curr):")
    for a,b,last_prev,first in examples:
        print(f" {a}->{b}: {last_prev} -> {first}")

# ---- summarize token counts per global id ----
all_ids = np.array(sorted(token_count.keys()), dtype=np.int64)
all_counts = np.array([token_count[int(i)] for i in all_ids], dtype=np.int64)

print("\n[token-per-segment-id summary]")
print("num unique segment ids (token side):", all_ids.size)
print("id range:", int(all_ids.min()), int(all_ids.max()))
print("token count stats (per id): min/mean/median/max =",
      int(all_counts.min()), float(all_counts.mean()), float(np.median(all_counts)), int(all_counts.max()))
print("percentiles:", np.percentile(all_counts, [0,5,25,50,75,95,100]).tolist())

# top 20 ids by allocated tokens
topk = 20
order = np.argsort(all_counts)[::-1][:topk]
print(f"\nTop-{topk} segment ids by allocated token-frames:")
for idx in order:
    print(f"  id={int(all_ids[idx])}: tokens={int(all_counts[idx])}")


num_shards: 100

[token-side boundaries k->k+1]
 first == last: 95 / 99
 first == last+1: 3 / 99
 other: 1 / 99

examples of 'other' boundaries (prev->curr: last_prev -> first_curr):
 31->32: 12693 -> 12695

[token-per-segment-id summary]
num unique segment ids (token side): 28644
id range: 0 38845
token count stats (per id): min/mean/median/max = 1 69.34405111018015 45.0 5270
percentiles: [1.0, 1.0, 20.0, 45.0, 77.0, 196.0, 5270.0]

Top-20 segment ids by allocated token-frames:
  id=16241: tokens=5270
  id=31058: tokens=4666
  id=31333: tokens=4256
  id=7314: tokens=3871
  id=16214: tokens=3810
  id=4085: tokens=3359
  id=31308: tokens=3271
  id=20777: tokens=3165
  id=11615: tokens=3060
  id=7716: tokens=3009
  id=23368: tokens=2984
  id=15223: tokens=2920
  id=31332: tokens=2633
  id=37589: tokens=2450
  id=31063: tokens=2383
  id=15221: tokens=2379
  id=16239: tokens=2365
  id=31365: tokens=2293
  id=7684: tokens=2222
  id=31265: tokens=2220


In [28]:
import json
from pathlib import Path
import numpy as np

root = Path("/root/work/data/raw/train_v2.0")
s = 32
meta_root = json.loads((root/"metadata.json").read_text())
num_shards = meta_root["num_shards"]

def load_seg_raw(k: int):
    meta = json.loads((root/"metadata"/f"metadata_{k}.json").read_text())
    n_raw = int(meta["shard_num_frames"])
    i = int(meta["shard_ind"])
    seg = np.memmap(root/"segment_indices"/f"segment_idx_{i}.bin",
                    dtype=np.int32, mode="r", shape=(n_raw,))
    return seg, n_raw, i

def get_n_tok(i: int):
    vp = root/"segment_indices"/"videos"/f"video_{i}.bin"
    vb = vp.stat().st_size
    return vb // (s*s*np.dtype(np.int32).itemsize)

def seg_tok_from_raw_index(seg_raw, raw_index):
    raw_index = np.clip(raw_index, 0, len(seg_raw)-1)
    return np.array(seg_raw[raw_index], dtype=np.int32)

# --- mapping candidates ---
def map_floor(n_raw, n_tok):
    t = np.arange(n_tok, dtype=np.int64)
    return (t * n_raw) // n_tok

def map_round(n_raw, n_tok):
    t = np.arange(n_tok, dtype=np.float64)
    return np.rint(t * (n_raw / n_tok)).astype(np.int64)

def map_fixed17_scaled(n_raw, n_tok):
    """
    “固定 17→3” を強引に n_raw と n_tok に合わせる版：
    基本形 floor(t*17/3) を作り、最後が n_raw-1 に合うよう線形スケール。
    （固定比そのままでは n_raw と整合しない可能性があるため）
    """
    t = np.arange(n_tok, dtype=np.float64)
    base = np.floor(t * (17.0/3.0))  # 固定17/3
    if base[-1] <= 0:
        return np.zeros(n_tok, dtype=np.int64)
    # 端点合わせスケール
    scaled = np.rint(base * ((n_raw-1) / base[-1])).astype(np.int64)
    # 単調化（安全）
    scaled = np.maximum.accumulate(scaled)
    return scaled

# --- statistics holders ---
ratios = []
need_drop_raw = []   # 固定17/3が正しいなら必要になりそうな raw 端数（+なら余る/捨てる、-なら足りない/パディング）
need_drop_tok = []   # 同様にtoken側

# boundary continuity counts for each mapping
same_counts = {"fixed17":0, "floor":0, "round":0}
plus1_counts = {"fixed17":0, "floor":0, "round":0}
other_counts = {"fixed17":0, "floor":0, "round":0}

prev_last = {"fixed17":None, "floor":None, "round":None}

for k in range(num_shards):
    seg_raw, n_raw, i = load_seg_raw(k)
    n_tok = get_n_tok(i)

    ratios.append(n_raw / n_tok)

    # “固定17/3”なら期待される n_tok と n_raw のズレ（どれだけ捨て/パディングが必要か）
    tok_from_raw = int(np.rint(n_raw * (3/17)))
    raw_from_tok = int(np.rint(n_tok * (17/3)))
    need_drop_tok.append(n_tok - tok_from_raw)
    need_drop_raw.append(n_raw - raw_from_tok)

    # build seg_tok for each mapping
    m_fixed = seg_tok_from_raw_index(seg_raw, map_fixed17_scaled(n_raw, n_tok))
    m_floor = seg_tok_from_raw_index(seg_raw, map_floor(n_raw, n_tok))
    m_round = seg_tok_from_raw_index(seg_raw, map_round(n_raw, n_tok))

    # boundary checks
    for name, arr in [("fixed17", m_fixed), ("floor", m_floor), ("round", m_round)]:
        if k > 0:
            first = int(arr[0])
            last_prev = int(prev_last[name])
            if first == last_prev:
                same_counts[name] += 1
            elif first == last_prev + 1:
                plus1_counts[name] += 1
            else:
                other_counts[name] += 1
        prev_last[name] = int(arr[-1])

ratios = np.array(ratios, dtype=np.float64)
need_drop_raw = np.array(need_drop_raw, dtype=np.int64)
need_drop_tok = np.array(need_drop_tok, dtype=np.int64)

print("=== ratio check ===")
print("mean n_raw/n_tok:", float(ratios.mean()))
print("mean diff from 17/3:", float((ratios - (17/3)).mean()))
print("abs diff percentiles:", np.percentile(np.abs(ratios - (17/3)), [0,50,90,95,99,100]).tolist())

print("\n=== if fixed 17/3, how much mismatch would need handling? ===")
print("need_drop_raw (n_raw - round(n_tok*17/3))  stats min/median/max:",
      int(need_drop_raw.min()), int(np.median(need_drop_raw)), int(need_drop_raw.max()))
print("need_drop_tok (n_tok - round(n_raw*3/17))  stats min/median/max:",
      int(need_drop_tok.min()), int(np.median(need_drop_tok)), int(need_drop_tok.max()))
print("top10 |need_drop_raw|:", np.sort(np.abs(need_drop_raw))[-10:].tolist())

print("\n=== token-side boundary continuity (k->k+1) ===")
for name in ["fixed17", "floor", "round"]:
    print(f"[{name:7s}] first==last: {same_counts[name]}/99, first==last+1: {plus1_counts[name]}/99, other: {other_counts[name]}/99")


=== ratio check ===
mean n_raw/n_tok: 5.66591753443089
mean diff from 17/3: -0.0007491322357792729
abs diff percentiles: [0.00015110305228205334, 0.0007551729346024771, 0.0007551729346024771, 0.0007551729346024771, 0.0007551729346024771, 0.0007551729346024771]

=== if fixed 17/3, how much mismatch would need handling? ===
need_drop_raw (n_raw - round(n_tok*17/3))  stats min/median/max: -15 -15 -3
need_drop_tok (n_tok - round(n_raw*3/17))  stats min/median/max: 1 3 3
top10 |need_drop_raw|: [15, 15, 15, 15, 15, 15, 15, 15, 15, 15]

=== token-side boundary continuity (k->k+1) ===
[fixed17] first==last: 97/99, first==last+1: 2/99, other: 0/99
[floor  ] first==last: 95/99, first==last+1: 3/99, other: 1/99
[round  ] first==last: 95/99, first==last+1: 3/99, other: 1/99


In [4]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

# ★あなたの環境のパスに合わせて変更
root = Path("/root/work/data/raw/train_v2.0")

meta_root = json.loads((root / "metadata.json").read_text())
num_shards = int(meta_root["num_shards"])
hz = int(meta_root.get("hz", 30))

def load_seg_for_shard(k: int):
    """
    shard k の segment_idx を memmap で開く
    """
    meta = json.loads((root / "metadata" / f"metadata_{k}.json").read_text())
    n_raw = int(meta["shard_num_frames"])
    shard_ind = int(meta["shard_ind"])
    seg_path = root / "segment_indices" / f"segment_idx_{shard_ind}.bin"
    seg = np.memmap(seg_path, dtype=np.int32, mode="r", shape=(n_raw,))
    return seg, shard_ind, n_raw

max_id = 100

counts = np.zeros(max_id + 1, dtype=np.int64)
first_shard = np.full(max_id + 1, -1, dtype=np.int32)
last_shard  = np.full(max_id + 1, -1, dtype=np.int32)

for k in range(num_shards):
    seg, shard_ind, n_raw = load_seg_for_shard(k)

    # shard内での (id -> 出現数) を一気に取る
    vals, cnt = np.unique(seg, return_counts=True)

    # 0..100だけ拾って足し込む
    mask = (vals >= 0) & (vals <= max_id)
    for v, c in zip(vals[mask], cnt[mask]):
        v = int(v); c = int(c)
        counts[v] += c
        if first_shard[v] == -1:
            first_shard[v] = k
        last_shard[v] = k

df = pd.DataFrame({
    "segment_id": np.arange(max_id + 1, dtype=int),
    "num_raw_frames": counts,
    "duration_sec": counts / float(hz),
    "first_shard": first_shard,
    "last_shard": last_shard,
    "spans_multiple_shards": (first_shard != -1) & (first_shard != last_shard),
})

# 0..100 は通常出るはずですが、念のため 0件は落とす
df = df[df["num_raw_frames"] > 0].reset_index(drop=True)

display(df)

# 参考：どのIDがファイルを跨いでいるかだけ見たい場合
display(df[df["spans_multiple_shards"]])


,segment_id,num_raw_frames,duration_sec,first_shard,last_shard,spans_multiple_shards
0,0,806,26.866667,0,0,False
1,1,970,32.333333,0,0,False
2,2,1103,36.766667,0,0,False
3,3,837,27.900000,0,0,False
4,4,1006,33.533333,0,0,False
...,...,...,...,...,...,...
96,96,245,8.166667,0,0,False
97,97,319,10.633333,0,0,False
98,98,611,20.366667,0,0,False
99,99,145,4.833333,0,0,False


,segment_id,num_raw_frames,duration_sec,first_shard,last_shard,spans_multiple_shards


In [13]:
seg, seg_ind, n_raw = load_seg_for_shard(0)



In [14]:
seg

memmap([  0,   0,   0, ..., 520, 520, 520], shape=(112542,), dtype=int32)

In [15]:
vals, cnt = np.unique(seg, return_counts=True)

In [16]:
print(vals.shape)
print(cnt.shape)


(521,)
(521,)


In [37]:
robot_state = np.memmap("/root/work/data/raw/train_v2.0/robot_states/states_0.bin", dtype=np.float32, mode="r", shape=(112542, 25))

In [41]:
robot_state[100]

memmap([-6.7175953e-03, -1.9181071e-02, -2.0101801e-01,  1.9568169e-01,
         1.5499594e-02,  1.6176233e-02,  2.3166141e-01,  1.1286174e-01,
        -1.4256638e-01, -1.2445254e+00,  1.5374298e-01,  1.7547835e-01,
         8.7971441e-02,  1.3495088e-01, -1.2821586e-01,  5.1873207e-01,
        -1.8492981e+00,  4.5381933e-02,  1.8340743e-01, -1.6123145e-03,
         3.0635832e-02,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00], dtype=float32)

In [17]:
vals

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

In [18]:
cnt

array([ 806,  970, 1103,  837, 1006, 1176, 1027, 1026,  188,    1,  320,
          1,  284,    1,  291,    1,  360,    1,  339,    1,  298,    1,
        385,  270,  256,  232,    4,  293,  230,    1,  164,   11,    6,
        178,   59,   15,  324,    1,  260,    1,  233,    1,  233,    1,
        116,  100,    1,  206,    1,  199,    1,  228,    1,  227,    1,
        187,    1,  211,    1,  223,  227,    1,  205,    1,  162,   23,
          1,  152,    8,  169,    1,  160,  730,  215,    1, 1624,  353,
         15,    1,  490,    1,  479, 1039,   54,    1,  194,    1, 1882,
          1,    3,  515,   18,  845,  348,   32,    4,  245,  319,  611,
        145,  104,  325,  263,   39,    7,  285,  134,  297,  409,  355,
         44,  753,  203,   67,  256,  203,  246, 1081,  565,  173,  173,
        161,    6,  312,  389,   21,   63,   89,  266,  163, 1461, 1211,
         67,   26,   29,   21,    3,    9,   28,   63,    7,   41,   10,
        367,  179,  179,  156,  199,  174,  560,  1

In [21]:
806 % 17

7

In [31]:
import json
from pathlib import Path
import numpy as np
from collections import defaultdict

root = Path("/root/work/data/raw/train_v2.0")
meta_root = json.loads((root/"metadata.json").read_text())
num_shards = int(meta_root["num_shards"])
hz = int(meta_root.get("hz", 30))
s = 32

def load_seg_raw(k: int):
    meta = json.loads((root/"metadata"/f"metadata_{k}.json").read_text())
    n_raw = int(meta["shard_num_frames"])
    i = int(meta["shard_ind"])
    seg = np.memmap(root/"segment_indices"/f"segment_idx_{i}.bin", dtype=np.int32, mode="r", shape=(n_raw,))
    return seg, n_raw, i

def get_n_tok(i: int):
    vp = root/"segment_indices"/"videos"/f"video_{i}.bin"
    vb = vp.stat().st_size
    return vb // (s*s*np.dtype(np.int32).itemsize)

def map_fixed17_scaled(n_raw, n_tok):
    # 固定 17/3 をベースにして端点合わせ（あなたが前に回した検証と同じ思想）
    t = np.arange(n_tok, dtype=np.float64)
    base = np.floor(t * (17.0/3.0))
    if base[-1] <= 0:
        return np.zeros(n_tok, dtype=np.int64)
    scaled = np.rint(base * ((n_raw-1) / base[-1])).astype(np.int64)
    scaled = np.maximum.accumulate(scaled)
    return np.clip(scaled, 0, n_raw-1)

raw_count = defaultdict(int)   # segment_id -> raw frames
tok_count = defaultdict(int)   # segment_id -> token frames (by seg_tok)

for k in range(num_shards):
    seg_raw, n_raw, i = load_seg_raw(k)
    n_tok = get_n_tok(i)

    # raw length per id (within shard; global accumulate)
    vals, cnt = np.unique(seg_raw, return_counts=True)
    for v, c in zip(vals.tolist(), cnt.tolist()):
        raw_count[int(v)] += int(c)

    # token -> raw index by fixed17 mapping, then seg_tok
    raw_index = map_fixed17_scaled(n_raw, n_tok)
    seg_tok = np.array(seg_raw[raw_index], dtype=np.int32)

    vals2, cnt2 = np.unique(seg_tok, return_counts=True)
    for v, c in zip(vals2.tolist(), cnt2.tolist()):
        tok_count[int(v)] += int(c)

# ---- evaluate hypotheses on ids that appear on token side ----
ids = sorted(tok_count.keys())
raw_len = np.array([raw_count[i] for i in ids], dtype=np.int64)
tok_len = np.array([tok_count[i] for i in ids], dtype=np.int64)

pred_8  = (raw_len + 8 - 1) // 8                 # ceil(raw/8)
pred_17 = (raw_len * 3 + 17 - 1) // 17           # ceil(raw*3/17)  (= ceil(raw/(17/3)))

acc_8  = float((tok_len == pred_8).mean())
acc_17 = float((tok_len == pred_17).mean())

mae_8  = float(np.mean(np.abs(tok_len - pred_8)))
mae_17 = float(np.mean(np.abs(tok_len - pred_17)))

print("num ids evaluated:", len(ids))
print("\n[Hypothesis A] tok_len == ceil(raw/8)")
print("  exact match rate:", acc_8)
print("  mean abs error  :", mae_8)

print("\n[Hypothesis B] tok_len == ceil(raw*3/17)")
print("  exact match rate:", acc_17)
print("  mean abs error  :", mae_17)

# 参考：比率の統計
ratio = raw_len / np.maximum(tok_len, 1)
print("\nraw/tok ratio stats (from counts):",
      "min/median/mean/max =",
      float(ratio.min()), float(np.median(ratio)), float(ratio.mean()), float(ratio.max()))

# どちらが外れているか例を見る
bad8 = np.where(tok_len != pred_8)[0]
bad17 = np.where(tok_len != pred_17)[0]
print("\nexamples where /8 fails (first 10):")
for j in bad8[:10]:
    print(" id", ids[j], "raw", int(raw_len[j]), "tok", int(tok_len[j]), "ceil(raw/8)", int(pred_8[j]))
print("\nexamples where 17->3 fails (first 10):")
for j in bad17[:10]:
    print(" id", ids[j], "raw", int(raw_len[j]), "tok", int(tok_len[j]), "ceil(raw*3/17)", int(pred_17[j]))


num ids evaluated: 28716

[Hypothesis A] tok_len == ceil(raw/8)
  exact match rate: 0.13393230254910155
  mean abs error  : 19.75818359102939

[Hypothesis B] tok_len == ceil(raw*3/17)
  exact match rate: 0.5670009750661652
  mean abs error  : 0.4329990249338348

raw/tok ratio stats (from counts): min/median/mean/max = 1.0 5.662921348314606 5.303565718671491 11.0

examples where /8 fails (first 10):
 id 0 raw 806 tok 143 ceil(raw/8) 101
 id 1 raw 970 tok 171 ceil(raw/8) 122
 id 2 raw 1103 tok 195 ceil(raw/8) 138
 id 3 raw 837 tok 147 ceil(raw/8) 105
 id 4 raw 1006 tok 178 ceil(raw/8) 126
 id 5 raw 1176 tok 207 ceil(raw/8) 147
 id 6 raw 1027 tok 182 ceil(raw/8) 129
 id 7 raw 1026 tok 181 ceil(raw/8) 129
 id 8 raw 188 tok 33 ceil(raw/8) 24
 id 10 raw 320 tok 57 ceil(raw/8) 40

examples where 17->3 fails (first 10):
 id 1 raw 970 tok 171 ceil(raw*3/17) 172
 id 3 raw 837 tok 147 ceil(raw*3/17) 148
 id 5 raw 1176 tok 207 ceil(raw*3/17) 208
 id 7 raw 1026 tok 181 ceil(raw*3/17) 182
 id 8 raw 

In [32]:
import json
from pathlib import Path
import numpy as np
from collections import defaultdict

root = Path("/root/work/data/raw/train_v2.0")
meta_root = json.loads((root/"metadata.json").read_text())
num_shards = int(meta_root["num_shards"])
hz = int(meta_root.get("hz", 30))
s = 32

# ---- shard lengths & paths ----
raw_lens = []
tok_lens = []
seg_paths = []
for k in range(num_shards):
    m = json.loads((root/"metadata"/f"metadata_{k}.json").read_text())
    n_raw = int(m["shard_num_frames"])
    i = int(m["shard_ind"])
    raw_lens.append(n_raw)
    seg_paths.append(root/"segment_indices"/f"segment_idx_{i}.bin")

    vp = root/"segment_indices"/"videos"/f"video_{i}.bin"
    vb = vp.stat().st_size
    n_tok = vb // (s*s*np.dtype(np.int32).itemsize)
    tok_lens.append(n_tok)

raw_lens = np.array(raw_lens, dtype=np.int64)
tok_lens = np.array(tok_lens, dtype=np.int64)

raw_cum = np.concatenate([[0], np.cumsum(raw_lens)])   # length num_shards+1
tok_cum = np.concatenate([[0], np.cumsum(tok_lens)])

N_raw_total = int(raw_cum[-1])
N_tok_total = int(tok_cum[-1])

print("N_raw_total:", N_raw_total, "N_tok_total:", N_tok_total, "ratio:", N_raw_total/N_tok_total)

# ---- helper: locate global raw index -> (shard, local_idx) ----
def locate_raw(global_idx: np.ndarray):
    # returns shard indices for each global_idx
    # shard = largest j such that raw_cum[j] <= idx
    shard = np.searchsorted(raw_cum, global_idx, side="right") - 1
    local = global_idx - raw_cum[shard]
    return shard.astype(np.int64), local.astype(np.int64)

# ---- raw_count: exact from raw segment_idx (sum over shards) ----
raw_count = defaultdict(int)
for k in range(num_shards):
    n = int(raw_lens[k])
    seg = np.memmap(seg_paths[k], dtype=np.int32, mode="r", shape=(n,))
    vals, cnt = np.unique(seg, return_counts=True)
    for v, c in zip(vals.tolist(), cnt.tolist()):
        raw_count[int(v)] += int(c)

# ---- tok_count: build seg_tok using GLOBAL mapping ----
tok_count = defaultdict(int)

# choose mapping:
# A) fixed 17->3 (raw per token = 17/3) with global time
def raw_index_fixed17(T_global: np.ndarray):
    return np.floor(T_global * (17.0/3.0)).astype(np.int64)

# B) generic proportional mapping (if you want)
def raw_index_prop(T_global: np.ndarray):
    return ((T_global * N_raw_total) // N_tok_total).astype(np.int64)

# Use fixed17 here (you can switch to raw_index_prop to compare)
use_fixed17 = True

chunk = 2_000_000  # process tokens in chunks to avoid huge arrays
T0 = 0
while T0 < N_tok_total:
    T1 = min(T0 + chunk, N_tok_total)
    T = np.arange(T0, T1, dtype=np.int64)

    if use_fixed17:
        raw_g = raw_index_fixed17(T)
    else:
        raw_g = raw_index_prop(T)

    # clip to valid raw range
    raw_g = np.clip(raw_g, 0, N_raw_total - 1)

    shard_idx, local_idx = locate_raw(raw_g)

    # read segment ids for this chunk (may span many shards)
    # group by shard for efficient reads
    order = np.argsort(shard_idx)
    shard_sorted = shard_idx[order]
    local_sorted = local_idx[order]

    start = 0
    while start < len(order):
        sh = int(shard_sorted[start])
        end = start
        while end < len(order) and int(shard_sorted[end]) == sh:
            end += 1

        n = int(raw_lens[sh])
        seg = np.memmap(seg_paths[sh], dtype=np.int32, mode="r", shape=(n,))
        ids = np.array(seg[local_sorted[start:end]], dtype=np.int32)

        # count
        vals, cnt = np.unique(ids, return_counts=True)
        for v, c in zip(vals.tolist(), cnt.tolist()):
            tok_count[int(v)] += int(c)

        start = end

    T0 = T1

# ---- evaluate hypotheses for per-segment lengths ----
ids = sorted(tok_count.keys())
raw_len = np.array([raw_count[i] for i in ids], dtype=np.int64)
tok_len = np.array([tok_count[i] for i in ids], dtype=np.int64)

pred_8  = (raw_len + 8 - 1) // 8
pred_17 = (raw_len * 3 + 17 - 1) // 17

acc_8  = float((tok_len == pred_8).mean())
acc_17 = float((tok_len == pred_17).mean())
mae_8  = float(np.mean(np.abs(tok_len - pred_8)))
mae_17 = float(np.mean(np.abs(tok_len - pred_17)))

print("\n[GLOBAL mapping results]")
print("tok_len == ceil(raw/8):    acc =", acc_8,  "MAE =", mae_8)
print("tok_len == ceil(raw*3/17): acc =", acc_17, "MAE =", mae_17)


N_raw_total: 11254161 N_tok_total: 1986291 ratio: 5.665917531721183

[GLOBAL mapping results]
tok_len == ceil(raw/8):    acc = 0.13013268156424582 MAE = 19.81340782122905
tok_len == ceil(raw*3/17): acc = 0.5633729050279329 MAE = 0.44574022346368714


In [35]:
import numpy as np

# すでにあなたが得た全長
N_raw_total = 11254161
N_tok_total = 1986291

# 固定17->3が本命なので、これで raw_index を作る
T = np.arange(N_tok_total, dtype=np.int64)
raw_g = np.floor(T * (17.0/3.0)).astype(np.int64)
raw_g = np.clip(raw_g, 0, N_raw_total-1)

# 6 tokenごとの境界（未来予測タスクのサンプル境界候補）
boundary_T = np.arange(0, N_tok_total, 6, dtype=np.int64)
raw_at_boundary = raw_g[boundary_T]

# rawの34境界（0,34,68,...）との差（剰余）を見る
rem34 = raw_at_boundary % 34
# 0に近いほど整列。0と33は同じ距離とみなすなら circular distance
dist34 = np.minimum(rem34, 34 - rem34)

print("N_tok_total%6:", N_tok_total % 6)
print("num boundaries:", len(boundary_T))

print("\nAlignment to raw 34-grid at 6-token boundaries")
print("dist34 stats min/median/mean/max:",
      int(dist34.min()), float(np.median(dist34)), float(dist34.mean()), int(dist34.max()))

# どれくらい0に集中してるか
for d in range(0, 6):  # 0..5 くらい見れば十分
    print(f"dist34=={d}: {(dist34==d).mean():.4f}")

# 参考：34境界に完全一致する割合
print("\nexact aligned (dist34==0) ratio:", float((dist34==0).mean()))


N_tok_total%6: 3
num boundaries: 331049

Alignment to raw 34-grid at 6-token boundaries
dist34 stats min/median/mean/max: 0 0.0 0.0013291083797262642 10
dist34==0: 0.9999
dist34==1: 0.0000
dist34==2: 0.0000
dist34==3: 0.0000
dist34==4: 0.0000
dist34==5: 0.0000

exact aligned (dist34==0) ratio: 0.9998670891620274


In [34]:
# あなたがすでに出してくれた値
N_tok_total = 1986291

print("N_tok_total % 6 =", N_tok_total % 6)
print("N_tok_total % 3 =", N_tok_total % 3)


N_tok_total % 6 = 3
N_tok_total % 3 = 0


In [2]:
import json
from pathlib import Path
from typing import List, Tuple, Dict, Any

import numpy as np
import torch
from torch.utils.data import Dataset as TorchDataset


class ShardedBlockTokenDataset(TorchDataset):
    """
    train_v2.0 の video shards (video_{i}.bin) をグローバル連結して、
    6 token (= past 3 + future 3) を1サンプルとして返す Dataset.

    - tokens: int32, shape=(T, 32, 32) per shard
    - global token index を shard/local に変換して必要な分だけ memmap から読む
    """

    def __init__(
        self,
        root: str | Path,
        s: int = 32,
        past_frames: int = 3,
        future_frames: int = 3,
        token_dtype: np.dtype = np.int32,
        output_format: str = "seq2seq",  # "seq2seq" or "causal_lm"
        drop_last_incomplete_block: bool = True,
        device: str | None = None,       # NoneならCPUテンソル
    ):
        """
        Args:
            root: /root/work/data/raw/train_v2.0
            s: token frame spatial size (default 32)
            past_frames: number of past token-frames (default 3)
            future_frames: number of future token-frames (default 3)
            token_dtype: dtype of stored tokens (default int32)
            output_format:
                - "seq2seq": input_ids=past, labels=future
                - "causal_lm": input_ids=all(6), labels=all(6) but past labels masked (-100)
            drop_last_incomplete_block:
                - True: 6フレームに満たない末尾は捨てる（推奨）
            device: torch device string (e.g. "cuda") or None
        """
        self.root = Path(root)
        self.s = int(s)
        self.past_frames = int(past_frames)
        self.future_frames = int(future_frames)
        self.block_frames = self.past_frames + self.future_frames
        self.token_dtype = np.dtype(token_dtype)
        assert output_format in ("seq2seq", "causal_lm")
        self.output_format = output_format
        self.drop_last = bool(drop_last_incomplete_block)
        self.device = device

        # ---- load root metadata ----
        meta_root = json.loads((self.root / "metadata.json").read_text())
        self.num_shards = int(meta_root["num_shards"])

        # ---- per-shard: shard_ind, raw_frames, token_frames, memmap shape ----
        self.shard_inds: List[int] = []
        self.n_tok_by_shard: List[int] = []
        self.video_paths: List[Path] = []

        # token length = bytes / (s*s*itemsize)
        bytes_per_frame = self.s * self.s * self.token_dtype.itemsize

        for k in range(self.num_shards):
            m = json.loads((self.root / "metadata" / f"metadata_{k}.json").read_text())
            shard_ind = int(m["shard_ind"])
            self.shard_inds.append(shard_ind)

            vp = self.root / "segment_indices" / "videos" / f"video_{shard_ind}.bin"
            if not vp.exists():
                raise FileNotFoundError(vp)
            self.video_paths.append(vp)

            vb = vp.stat().st_size
            if vb % bytes_per_frame != 0:
                raise ValueError(f"{vp} size not divisible by bytes_per_frame={bytes_per_frame}: {vb}")
            n_tok = vb // bytes_per_frame
            self.n_tok_by_shard.append(int(n_tok))

        self.tok_cum = np.concatenate([[0], np.cumsum(self.n_tok_by_shard, dtype=np.int64)])
        self.N_tok_total = int(self.tok_cum[-1])

        # number of full blocks
        if self.drop_last:
            self.num_blocks = self.N_tok_total // self.block_frames
        else:
            # ceil (rarely useful here)
            self.num_blocks = (self.N_tok_total + self.block_frames - 1) // self.block_frames

        # lazy memmaps cache
        self._mmaps: Dict[int, np.memmap] = {}

    def __len__(self) -> int:
        return int(self.num_blocks)

    def _get_mmap(self, shard_k: int) -> np.memmap:
        """
        shard_k: 0..num_shards-1 (metadata index order)
        """
        if shard_k not in self._mmaps:
            vp = self.video_paths[shard_k]
            n_tok = self.n_tok_by_shard[shard_k]
            mm = np.memmap(
                vp,
                dtype=self.token_dtype,
                mode="r",
                shape=(n_tok, self.s, self.s),
            )
            self._mmaps[shard_k] = mm
        return self._mmaps[shard_k]

    def _locate(self, g: int) -> Tuple[int, int]:
        """
        global token index g -> (shard_k, local_index)
        """
        # shard_k is such that tok_cum[shard_k] <= g < tok_cum[shard_k+1]
        shard_k = int(np.searchsorted(self.tok_cum, g, side="right") - 1)
        local = int(g - self.tok_cum[shard_k])
        return shard_k, local

    def _read_range(self, g0: int, length: int) -> np.ndarray:
        """
        read token frames [g0, g0+length) across shards if needed.
        returns np.ndarray shape=(length, s, s) dtype=token_dtype
        """
        if length <= 0:
            return np.empty((0, self.s, self.s), dtype=self.token_dtype)

        out = np.empty((length, self.s, self.s), dtype=self.token_dtype)
        pos = 0
        g = g0

        while pos < length:
            shard_k, local = self._locate(g)
            mm = self._get_mmap(shard_k)
            remain_in_shard = self.n_tok_by_shard[shard_k] - local
            take = min(length - pos, remain_in_shard)
            out[pos:pos+take] = mm[local:local+take]
            pos += take
            g += take

            # drop_last=False の場合、末尾が足りなければ pad する
            if (not self.drop_last) and (g >= self.N_tok_total) and (pos < length):
                # pad with zeros
                out[pos:] = 0
                break

        return out

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        """
        idx: block index (0..num_blocks-1)
        """
        idx = int(idx)
        g0 = idx * self.block_frames

        frames = self._read_range(g0, self.block_frames)  # (6,32,32)
        frames_i64 = frames.astype(np.int64, copy=False)  # torch long

        # flatten per sample
        flat = torch.from_numpy(frames_i64.reshape(-1))  # (6*1024,)

        if self.output_format == "seq2seq":
            past = flat[: self.past_frames * self.s * self.s]
            fut  = flat[self.past_frames * self.s * self.s :]

            attention_mask = torch.ones_like(past)

            batch = {
                "input_ids": past,
                "labels": fut,
                "attention_mask": attention_mask,
                # 解析やデバッグ用に残しておくと便利
                "past_frames": frames_i64[: self.past_frames],   # (3,32,32) numpy (int64)
                "future_frames": frames_i64[self.past_frames :], # (3,32,32) numpy (int64)
            }

        else:  # causal_lm
            # labels は未来部分だけ loss を当てたいので past 部分を -100 にする
            labels = flat.clone()
            n_past = self.past_frames * self.s * self.s
            labels[:n_past] = -100

            attention_mask = torch.ones_like(flat)

            batch = {
                "input_ids": flat,
                "labels": labels,
                "attention_mask": attention_mask,
            }

        if self.device is not None:
            for k, v in list(batch.items()):
                if torch.is_tensor(v):
                    batch[k] = v.to(self.device)

        return batch


In [42]:
import json
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import torch
from torch.utils.data import Dataset as TorchDataset


class ShardedSlidingBlockDataset(TorchDataset):
    """
    - video shards をグローバル連結
    - window = 6 token (past3 + future3)
    - stride = 3 token（= 3トークン単位でスライド）
    - segment_idx (raw) を用いて、境界混入する3トークンブロックを除外
      さらに pastブロックと futureブロックが同一clipのものだけ残す
    """

    def __init__(
        self,
        root: str | Path,
        s: int = 32,
        past_frames: int = 3,
        future_frames: int = 3,
        stride_tokens: int = 3,                 # ← ここが 0-5,3-8,... の鍵
        token_dtype: np.dtype = np.int32,       # video_*.bin の dtype
        seg_dtype: np.dtype = np.int32,         # segment_idx_*.bin の dtype
        output_format: str = "seq2seq",         # "seq2seq" or "causal_lm"
        device: str | None = None,
        cache_path: str | Path | None = None,   # valid_starts を保存/ロード
        use_fixed_17to3_mapping: bool = True,   # Trueなら raw=floor(tok*17/3)
        chunk_tokens: int = 1_000_000,          # 前計算のチャンク
    ):
        self.root = Path(root)
        self.s = int(s)
        self.past_frames = int(past_frames)
        self.future_frames = int(future_frames)
        self.block_frames = self.past_frames + self.future_frames  # 6
        self.stride_tokens = int(stride_tokens)                    # 3
        assert self.block_frames == 6, "この実装は past3+future3(=6) 前提（必要なら一般化可）"
        assert self.stride_tokens == 3, "この実装は stride=3 前提（必要なら一般化可）"

        self.token_dtype = np.dtype(token_dtype)
        self.seg_dtype = np.dtype(seg_dtype)
        assert output_format in ("seq2seq", "causal_lm")
        self.output_format = output_format
        self.device = device
        self.use_fixed = use_fixed_17to3_mapping
        self.chunk_tokens = int(chunk_tokens)

        # ---- root metadata ----
        meta_root = json.loads((self.root / "metadata.json").read_text())
        self.num_shards = int(meta_root["num_shards"])
        self.hz = int(meta_root.get("hz", 30))

        # ---- shard paths & lengths (token) ----
        self.video_paths: List[Path] = []
        self.n_tok_by_shard: List[int] = []

        bytes_per_token_frame = self.s * self.s * self.token_dtype.itemsize

        for k in range(self.num_shards):
            m = json.loads((self.root / "metadata" / f"metadata_{k}.json").read_text())
            shard_ind = int(m["shard_ind"])
            vp = self.root / "segment_indices" / "videos" / f"video_{shard_ind}.bin"
            if not vp.exists():
                raise FileNotFoundError(vp)
            self.video_paths.append(vp)

            vb = vp.stat().st_size
            if vb % bytes_per_token_frame != 0:
                raise ValueError(f"{vp} size not divisible by {bytes_per_token_frame}: {vb}")
            self.n_tok_by_shard.append(int(vb // bytes_per_token_frame))

        self.tok_cum = np.concatenate([[0], np.cumsum(self.n_tok_by_shard, dtype=np.int64)])
        self.N_tok_total = int(self.tok_cum[-1])

        # ---- shard paths & lengths (raw segment_idx) ----
        self.seg_paths: List[Path] = []
        self.n_raw_by_shard: List[int] = []

        for k in range(self.num_shards):
            m = json.loads((self.root / "metadata" / f"metadata_{k}.json").read_text())
            shard_ind = int(m["shard_ind"])
            n_raw = int(m["shard_num_frames"])
            sp = self.root / "segment_indices" / f"segment_idx_{shard_ind}.bin"
            if not sp.exists():
                raise FileNotFoundError(sp)
            self.seg_paths.append(sp)
            self.n_raw_by_shard.append(n_raw)

        self.raw_cum = np.concatenate([[0], np.cumsum(self.n_raw_by_shard, dtype=np.int64)])
        self.N_raw_total = int(self.raw_cum[-1])

        # ---- memmap caches ----
        self._video_mmaps: Dict[int, np.memmap] = {}
        self._seg_mmaps: Dict[int, np.memmap] = {}

        # ---- build valid_starts (token indices) ----
        self.cache_path = Path(cache_path) if cache_path is not None else None
        self.valid_starts = self._build_or_load_valid_starts()

    def __len__(self) -> int:
        return int(len(self.valid_starts))

    # ---------- memmap helpers ----------
    def _get_video_mmap(self, shard_k: int) -> np.memmap:
        if shard_k not in self._video_mmaps:
            vp = self.video_paths[shard_k]
            n_tok = self.n_tok_by_shard[shard_k]
            self._video_mmaps[shard_k] = np.memmap(
                vp, dtype=self.token_dtype, mode="r", shape=(n_tok, self.s, self.s)
            )
        return self._video_mmaps[shard_k]

    def _get_seg_mmap(self, shard_k: int) -> np.memmap:
        if shard_k not in self._seg_mmaps:
            sp = self.seg_paths[shard_k]
            n_raw = self.n_raw_by_shard[shard_k]
            self._seg_mmaps[shard_k] = np.memmap(
                sp, dtype=self.seg_dtype, mode="r", shape=(n_raw,)
            )
        return self._seg_mmaps[shard_k]

    def _locate_token(self, g: int) -> Tuple[int, int]:
        shard_k = int(np.searchsorted(self.tok_cum, g, side="right") - 1)
        local = int(g - self.tok_cum[shard_k])
        return shard_k, local

    def _locate_raw(self, r: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        shard = np.searchsorted(self.raw_cum, r, side="right") - 1
        local = r - self.raw_cum[shard]
        return shard.astype(np.int64), local.astype(np.int64)

    def _read_token_range(self, g0: int, length: int) -> np.ndarray:
        out = np.empty((length, self.s, self.s), dtype=self.token_dtype)
        pos = 0
        g = g0
        while pos < length:
            shard_k, local = self._locate_token(g)
            mm = self._get_video_mmap(shard_k)
            remain = self.n_tok_by_shard[shard_k] - local
            take = min(length - pos, remain)
            out[pos:pos+take] = mm[local:local+take]
            pos += take
            g += take
        return out

    # ---------- build valid starts ----------
    def _build_or_load_valid_starts(self) -> np.ndarray:
        if self.cache_path is not None and self.cache_path.exists():
            return np.load(self.cache_path)

        # token -> raw mapping
        # fixed 17->3: raw_idx = floor(tok * 17/3)
        # ただし範囲を clip
        def token_to_raw(tok_idx: np.ndarray) -> np.ndarray:
            if self.use_fixed:
                r = np.floor(tok_idx * (17.0 / 3.0)).astype(np.int64)
            else:
                # 比例写像（代替）
                r = ((tok_idx * self.N_raw_total) // self.N_tok_total).astype(np.int64)
            return np.clip(r, 0, self.N_raw_total - 1)

        N = self.N_tok_total
        assert N >= 6

        # 3トークンブロック数（最後の端数は落とす：混入防止のため）
        n_groups = N // 3

        # group_valid[g] : 3トークンが同一clipか？
        # group_seg[g]   : そのclip_id（validの場合）
        group_valid = np.zeros(n_groups, dtype=np.bool_)
        group_seg = np.full(n_groups, -1, dtype=np.int32)

        # token index を 0..3*n_groups-1 だけ見る（端数は捨て）
        TokN = 3 * n_groups

        # chunk で segment_id を引く
        # seg_id[t] = segment_idx[ raw_idx(t) ]
        seg_id = np.empty(TokN, dtype=np.int32)

        t0 = 0
        while t0 < TokN:
            t1 = min(t0 + self.chunk_tokens, TokN)
            t = np.arange(t0, t1, dtype=np.int64)
            r = token_to_raw(t)
            shard, local = self._locate_raw(r)

            # shardごとにまとめて読んで埋める
            order = np.argsort(shard)
            shard_s = shard[order]
            local_s = local[order]
            out = np.empty(t1 - t0, dtype=np.int32)

            p = 0
            while p < len(order):
                sh = int(shard_s[p])
                q = p
                while q < len(order) and int(shard_s[q]) == sh:
                    q += 1
                mm = self._get_seg_mmap(sh)
                out_block = np.array(mm[local_s[p:q]], dtype=np.int32)
                out[p:q] = out_block
                p = q

            # 元の順序に戻す
            inv = np.empty_like(order)
            inv[order] = np.arange(len(order))
            seg_id[t0:t1] = out[inv]

            t0 = t1

        # 3つずつ見て、全部同じならvalid
        seg3 = seg_id.reshape(n_groups, 3)
        same01 = (seg3[:, 0] == seg3[:, 1])
        same12 = (seg3[:, 1] == seg3[:, 2])
        group_valid[:] = (same01 & same12)
        group_seg[:] = seg3[:, 0]

        # 次に「学習例（2 group = 6 token）」として valid start group を作る
        # start group g は g と g+1 が valid かつ clip_id が同じ
        ok = group_valid[:-1] & group_valid[1:] & (group_seg[:-1] == group_seg[1:])
        valid_group_starts = np.nonzero(ok)[0]  # group index

        # stride=3 なので token start = g*3 がちょうど 0-5,3-8,... になる
        valid_starts = (valid_group_starts.astype(np.int64) * 3)

        if self.cache_path is not None:
            self.cache_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(self.cache_path, valid_starts)

        return valid_starts

    # ---------- dataset output ----------
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        g0 = int(self.valid_starts[int(idx)])  # token start index (multiple of 3)
        frames = self._read_token_range(g0, 6).astype(np.int64, copy=False)  # (6,32,32)

        flat = torch.from_numpy(frames.reshape(-1))  # (6144,)

        if self.output_format == "seq2seq":
            past = flat[: 3 * self.s * self.s]          # 0..3071
            fut = flat[3 * self.s * self.s :]           # 3072..6143
            attn = torch.ones_like(past)
            batch = {
                "input_ids": past,
                "labels": fut,
                "attention_mask": attn,
                "past_frames": frames[:3],      # (3,32,32) np int64
                "future_frames": frames[3:],    # (3,32,32) np int64
                "token_start": g0,              # デバッグ用
            }
        else:
            labels = flat.clone()
            labels[: 3 * self.s * self.s] = -100
            attn = torch.ones_like(flat)
            batch = {
                "input_ids": flat,
                "labels": labels,
                "attention_mask": attn,
                "token_start": g0,
            }

        if self.device is not None:
            for k, v in list(batch.items()):
                if torch.is_tensor(v):
                    batch[k] = v.to(self.device)

        return batch


In [46]:
import json
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import torch
from torch.utils.data import Dataset as TorchDataset


class ShardedSlidingBlockDataset(TorchDataset):
    """
    - video shards をグローバル連結
    - window = 6 token (past3 + future3)
    - stride = 3 token（= 3トークン単位でスライド）
    - segment_idx (raw) を用いて、境界混入する3トークンブロックを除外
      さらに pastブロックと futureブロックが同一clipのものだけ残す
    - 追加: robot_states/states_{i}.bin から raw 34 (=17+17) フレーム分の状態(25次元)も返す
    """

    def __init__(
        self,
        root: str | Path,
        s: int = 32,
        past_frames: int = 3,
        future_frames: int = 3,
        stride_tokens: int = 3,                 # 0-5,3-8,... を作る
        token_dtype: np.dtype = np.int32,       # video_*.bin の dtype
        seg_dtype: np.dtype = np.int32,         # segment_idx_*.bin の dtype
        state_dtype: np.dtype = np.float32,     # states_*.bin の dtype
        state_dim: int = 25,                    # states の次元
        output_format: str = "seq2seq",         # "seq2seq" or "causal_lm"
        device: str | None = None,
        cache_path: str | Path | None = None,   # valid_starts を保存/ロード
        use_fixed_17to3_mapping: bool = True,   # raw=floor(tok*17/3)
        chunk_tokens: int = 1_000_000,          # 前計算のチャンク
    ):
        self.root = Path(root)
        self.s = int(s)
        self.past_frames = int(past_frames)
        self.future_frames = int(future_frames)
        self.block_frames = self.past_frames + self.future_frames  # 6
        self.stride_tokens = int(stride_tokens)                    # 3
        assert self.block_frames == 6, "この実装は past3+future3(=6) 前提"
        assert self.stride_tokens == 3, "この実装は stride=3 前提"

        self.token_dtype = np.dtype(token_dtype)
        self.seg_dtype = np.dtype(seg_dtype)
        self.state_dtype = np.dtype(state_dtype)
        self.state_dim = int(state_dim)

        assert output_format in ("seq2seq", "causal_lm")
        self.output_format = output_format
        self.device = device
        self.use_fixed = use_fixed_17to3_mapping
        self.chunk_tokens = int(chunk_tokens)

        # ---- root metadata ----
        meta_root = json.loads((self.root / "metadata.json").read_text())
        self.num_shards = int(meta_root["num_shards"])
        self.hz = int(meta_root.get("hz", 30))

        # ---- shard paths & lengths (token) ----
        self.video_paths: List[Path] = []
        self.n_tok_by_shard: List[int] = []

        bytes_per_token_frame = self.s * self.s * self.token_dtype.itemsize

        for k in range(self.num_shards):
            m = json.loads((self.root / "metadata" / f"metadata_{k}.json").read_text())
            shard_ind = int(m["shard_ind"])
            vp = self.root / "segment_indices" / "videos" / f"video_{shard_ind}.bin"
            if not vp.exists():
                raise FileNotFoundError(vp)
            self.video_paths.append(vp)

            vb = vp.stat().st_size
            if vb % bytes_per_token_frame != 0:
                raise ValueError(f"{vp} size not divisible by {bytes_per_token_frame}: {vb}")
            self.n_tok_by_shard.append(int(vb // bytes_per_token_frame))

        self.tok_cum = np.concatenate([[0], np.cumsum(self.n_tok_by_shard, dtype=np.int64)])
        self.N_tok_total = int(self.tok_cum[-1])

        # ---- shard paths & lengths (raw segment_idx & robot states) ----
        self.seg_paths: List[Path] = []
        self.state_paths: List[Path] = []
        self.n_raw_by_shard: List[int] = []

        bytes_per_state_frame = self.state_dim * self.state_dtype.itemsize  # 25*4=100 bytes/frame

        for k in range(self.num_shards):
            m = json.loads((self.root / "metadata" / f"metadata_{k}.json").read_text())
            shard_ind = int(m["shard_ind"])
            n_raw = int(m["shard_num_frames"])
            self.n_raw_by_shard.append(n_raw)

            sp = self.root / "segment_indices" / f"segment_idx_{shard_ind}.bin"
            if not sp.exists():
                raise FileNotFoundError(sp)
            self.seg_paths.append(sp)

            stp = self.root / "robot_states" / f"states_{shard_ind}.bin"
            if not stp.exists():
                raise FileNotFoundError(stp)

            # サイズチェック（任意だけど安全）
            sb = stp.stat().st_size
            need = n_raw * bytes_per_state_frame
            if sb != need:
                raise ValueError(f"{stp} size mismatch: on_disk={sb}, need={need} (n_raw={n_raw}, state_dim={self.state_dim})")

            self.state_paths.append(stp)

        self.raw_cum = np.concatenate([[0], np.cumsum(self.n_raw_by_shard, dtype=np.int64)])
        self.N_raw_total = int(self.raw_cum[-1])

        # ---- memmap caches ----
        self._video_mmaps: Dict[int, np.memmap] = {}
        self._seg_mmaps: Dict[int, np.memmap] = {}
        self._state_mmaps: Dict[int, np.memmap] = {}

        # ---- build valid_starts (token indices) ----
        self.cache_path = Path(cache_path) if cache_path is not None else None
        self.valid_starts = self._build_or_load_valid_starts()

    def __len__(self) -> int:
        return int(len(self.valid_starts))

    # ---------- memmap helpers ----------
    def _get_video_mmap(self, shard_k: int) -> np.memmap:
        if shard_k not in self._video_mmaps:
            vp = self.video_paths[shard_k]
            n_tok = self.n_tok_by_shard[shard_k]
            self._video_mmaps[shard_k] = np.memmap(
                vp, dtype=self.token_dtype, mode="r", shape=(n_tok, self.s, self.s)
            )
        return self._video_mmaps[shard_k]

    def _get_seg_mmap(self, shard_k: int) -> np.memmap:
        if shard_k not in self._seg_mmaps:
            sp = self.seg_paths[shard_k]
            n_raw = self.n_raw_by_shard[shard_k]
            self._seg_mmaps[shard_k] = np.memmap(
                sp, dtype=self.seg_dtype, mode="r", shape=(n_raw,)
            )
        return self._seg_mmaps[shard_k]

    def _get_state_mmap(self, shard_k: int) -> np.memmap:
        if shard_k not in self._state_mmaps:
            stp = self.state_paths[shard_k]
            n_raw = self.n_raw_by_shard[shard_k]
            self._state_mmaps[shard_k] = np.memmap(
                stp, dtype=self.state_dtype, mode="r", shape=(n_raw, self.state_dim)
            )
        return self._state_mmaps[shard_k]

    def _locate_token(self, g: int) -> Tuple[int, int]:
        shard_k = int(np.searchsorted(self.tok_cum, g, side="right") - 1)
        local = int(g - self.tok_cum[shard_k])
        return shard_k, local

    def _locate_raw(self, r: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        shard = np.searchsorted(self.raw_cum, r, side="right") - 1
        local = r - self.raw_cum[shard]
        return shard.astype(np.int64), local.astype(np.int64)

    def _read_token_range(self, g0: int, length: int) -> np.ndarray:
        out = np.empty((length, self.s, self.s), dtype=self.token_dtype)
        pos = 0
        g = g0
        while pos < length:
            shard_k, local = self._locate_token(g)
            mm = self._get_video_mmap(shard_k)
            remain = self.n_tok_by_shard[shard_k] - local
            take = min(length - pos, remain)
            out[pos:pos+take] = mm[local:local+take]
            pos += take
            g += take
        return out

    def _read_state_range_raw(self, r0: int, length: int) -> np.ndarray:
        """
        raw index r0 から length フレーム分の robot state (25次元) を shard を跨いで読む
        return: (length, 25) float32
        """
        out = np.empty((length, self.state_dim), dtype=self.state_dtype)
        pos = 0
        r = int(r0)
        while pos < length:
            shard_k = int(np.searchsorted(self.raw_cum, r, side="right") - 1)
            local = int(r - self.raw_cum[shard_k])

            mm = self._get_state_mmap(shard_k)
            remain = self.n_raw_by_shard[shard_k] - local
            take = min(length - pos, remain)

            out[pos:pos+take] = mm[local:local+take]
            pos += take
            r += take
        return out

    # ---------- build valid starts ----------
    def _build_or_load_valid_starts(self) -> np.ndarray:
        if self.cache_path is not None and self.cache_path.exists():
            return np.load(self.cache_path)

        def token_to_raw(tok_idx: np.ndarray) -> np.ndarray:
            if self.use_fixed:
                r = np.floor(tok_idx * (17.0 / 3.0)).astype(np.int64)
            else:
                r = ((tok_idx * self.N_raw_total) // self.N_tok_total).astype(np.int64)
            return np.clip(r, 0, self.N_raw_total - 1)

        N = self.N_tok_total
        n_groups = N // 3                # 3-token group 数（端数は捨てる）
        TokN = 3 * n_groups              # 判定に使う token 数

        seg_id = np.empty(TokN, dtype=np.int32)

        t0 = 0
        while t0 < TokN:
            t1 = min(t0 + self.chunk_tokens, TokN)
            t = np.arange(t0, t1, dtype=np.int64)
            r = token_to_raw(t)
            shard, local = self._locate_raw(r)

            order = np.argsort(shard)
            shard_s = shard[order]
            local_s = local[order]
            out = np.empty(t1 - t0, dtype=np.int32)

            p = 0
            while p < len(order):
                sh = int(shard_s[p])
                q = p
                while q < len(order) and int(shard_s[q]) == sh:
                    q += 1
                mm = self._get_seg_mmap(sh)
                out[p:q] = np.array(mm[local_s[p:q]], dtype=np.int32)
                p = q

            inv = np.empty_like(order)
            inv[order] = np.arange(len(order))
            seg_id[t0:t1] = out[inv]
            t0 = t1

        seg3 = seg_id.reshape(n_groups, 3)
        group_valid = (seg3[:, 0] == seg3[:, 1]) & (seg3[:, 1] == seg3[:, 2])
        group_seg = seg3[:, 0]

        # 6-token sample (= 2 groups) が clip を跨がない条件
        ok = group_valid[:-1] & group_valid[1:] & (group_seg[:-1] == group_seg[1:])
        valid_group_starts = np.nonzero(ok)[0]

        valid_starts = (valid_group_starts.astype(np.int64) * 3)  # token start index

        if self.cache_path is not None:
            self.cache_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(self.cache_path, valid_starts)

        return valid_starts

    # ---------- dataset output ----------
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        g0 = int(self.valid_starts[int(idx)])  # token start index, multiple of 3
        frames = self._read_token_range(g0, 6).astype(np.int64, copy=False)  # (6,32,32)

        # ★ raw 開始は 3-token group index k = g0//3 に対して raw_start = 17*k
        k = g0 // 3
        raw_start = int(17 * k)
        # past17 + future17 = 34
        states_34 = self._read_state_range_raw(raw_start, 34).astype(np.float32, copy=False)  # (34,25)
        states_past = states_34[:17]
        states_future = states_34[17:]

        flat = torch.from_numpy(frames.reshape(-1))  # (6144,)

        if self.output_format == "seq2seq":
            past = flat[: 3 * self.s * self.s]
            fut = flat[3 * self.s * self.s :]
            attn = torch.ones_like(past)

            batch = {
                "input_ids": past,
                "labels": fut,
                "attention_mask": attn,

                # token側（デバッグ用）
                "past_frames": frames[:3],      # (3,32,32) np int64
                "future_frames": frames[3:],    # (3,32,32) np int64
                "token_start": g0,

                # ★ 追加：robot states
                "robot_states": states_34,          # (34,25) np float32
                "robot_states_past": states_past,   # (17,25)
                "robot_states_future": states_future,# (17,25)
                "raw_start": raw_start,             # デバッグ用
            }
        else:
            labels = flat.clone()
            labels[: 3 * self.s * self.s] = -100
            attn = torch.ones_like(flat)
            batch = {
                "input_ids": flat,
                "labels": labels,
                "attention_mask": attn,
                "token_start": g0,

                "robot_states": states_34,
                "robot_states_past": states_past,
                "robot_states_future": states_future,
                "raw_start": raw_start,
            }

        if self.device is not None:
            for k2, v2 in list(batch.items()):
                if torch.is_tensor(v2):
                    batch[k2] = v2.to(self.device)

        return batch


In [54]:
ds = ShardedSlidingBlockDataset(
    root="/root/work/data/raw/train_v2.0",
    output_format="seq2seq",
    cache_path=None,
)

x = ds[46]
print(x["input_ids"].shape, x["labels"].shape)          # torch.Size([3072]) torch.Size([3072])
print(x["robot_states"].shape)                          # (34,25)
print(x["robot_states_past"].shape, x["robot_states_future"].shape)  # (17,25) (17,25)
print("token_start:", x["token_start"], "raw_start:", x["raw_start"])


torch.Size([3072]) torch.Size([3072])
(34, 25)
(17, 25) (17, 25)
token_start: 144 raw_start: 816


In [55]:
x = ds[47]
print(x["input_ids"].shape, x["labels"].shape)          # torch.Size([3072]) torch.Size([3072])
print(x["robot_states"].shape)                          # (34,25)
print(x["robot_states_past"].shape, x["robot_states_future"].shape)  # (17,25) (17,25)
print("token_start:", x["token_start"], "raw_start:", x["raw_start"])

torch.Size([3072]) torch.Size([3072])
(34, 25)
(17, 25) (17, 25)
token_start: 147 raw_start: 833


In [2]:
import os, sys
import torch
import numpy as np

# ---- 1) import Cosmos Tokenizer ----
sys.path.append("/root/work/src/Cosmos-Tokenizer/")
from cosmos_tokenizer.video_lib import CausalVideoTokenizer

model_name = "Cosmos-Tokenizer-DV8x8x8"
dec_ckpt = f"/root/work/src/Cosmos-Tokenizer/pretrained_ckpts/{model_name}/decoder.jit"

decoder = CausalVideoTokenizer(checkpoint_dec=dec_ckpt)

# ---- 2) build dataset and pick one sample ----
ds = ShardedBlockTokenDataset(
    root="/root/work/data/raw/train_v2.0",
    output_format="seq2seq",
    drop_last_incomplete_block=True,
    device=None,
)

idx = 0  # 適当に変えてOK
sample = ds[idx]

# past_frames: numpy int64 shape (3,32,32)
past = sample["past_frames"]
print("past_frames np:", type(past), past.shape, past.dtype, "min/max:", int(past.min()), int(past.max()))

# ---- 3) prepare indices tensor (try multiple layouts) ----
# decoderが想定する dtype はだいたい int64/long が無難
past_t = torch.from_numpy(past.astype(np.int64, copy=False)).to("cuda")

candidates = [
    ("B,T,H,W", past_t.unsqueeze(0)),                 # (1,3,32,32)
    ("B,1,T,H,W", past_t.unsqueeze(0).unsqueeze(1)),  # (1,1,3,32,32)
    ("B,H,W,T", past_t.permute(1,2,0).unsqueeze(0)),  # (1,32,32,3)
    ("B,1,H,W,T", past_t.permute(1,2,0).unsqueeze(0).unsqueeze(1)),  # (1,1,32,32,3)
]

ok = False
for name, indices in candidates:
    try:
        with torch.no_grad():
            recon = decoder.decode(indices)
        print(f"\n✅ decode succeeded with layout: {name}")
        print("indices:", tuple(indices.shape), indices.dtype, indices.device)
        print("recon  :", tuple(recon.shape), recon.dtype, recon.device, "min/max:", float(recon.min()), float(recon.max()))
        ok = True
        break
    except Exception as e:
        print(f"\n❌ decode failed with layout: {name}")
        print("  indices shape:", tuple(indices.shape), "dtype:", indices.dtype)
        print("  error:", repr(e))

if not ok:
    raise RuntimeError("All candidate index layouts failed. See errors above for the expected shape.")


past_frames np: <class 'numpy.ndarray'> (3, 32, 32) int64 min/max: 124 63843

✅ decode succeeded with layout: B,T,H,W
indices: (1, 3, 32, 32) torch.int64 cuda:0
recon  : (1, 3, 17, 256, 256) torch.bfloat16 cuda:0 min/max: -1.03125 1.2890625


In [7]:
import os, sys, json
from pathlib import Path
import numpy as np
import torch
from IPython.display import Video, display
import imageio.v2 as imageio
# --- あなたのDataset（そのまま貼ったクラスを事前に定義済み前提） ---
# from your_cell import ShardedBlockTokenDataset

# --- Cosmos Tokenizer import ---
sys.path.append(os.path.join("/root/work/src/Cosmos-Tokenizer/"))
from cosmos_tokenizer.video_lib import CausalVideoTokenizer

# ----------------------------
# 設定
# ----------------------------
root = "/root/work/data/raw/train_v2.0"
model_name = "Cosmos-Tokenizer-DV8x8x8"
dec_ckpt = f"/root/work/src/Cosmos-Tokenizer/pretrained_ckpts/{model_name}/decoder.jit"

sample_idx = 2
fps = 30

# ----------------------------
# 1) サンプル取得（past/future: (3,32,32) int64）
# ----------------------------
ds = ShardedBlockTokenDataset(root=root, output_format="seq2seq", drop_last_incomplete_block=True, device=None)
sample = ds[sample_idx]
past_tok = sample["past_frames"]    # np.ndarray (3,32,32)
fut_tok  = sample["future_frames"]  # np.ndarray (3,32,32)

# ----------------------------
# 2) decoder.decode の入力shapeを複数試す
# ----------------------------
decoder = CausalVideoTokenizer(checkpoint_dec=dec_ckpt)

def decode_one(tok_3x32x32: np.ndarray):
    t = torch.from_numpy(tok_3x32x32.astype(np.int64, copy=False)).to("cuda")
    candidates = [
        ("B,T,H,W",      t.unsqueeze(0)),                 # (1,3,32,32)
        ("B,1,T,H,W",    t.unsqueeze(0).unsqueeze(1)),    # (1,1,3,32,32)
        ("B,H,W,T",      t.permute(1,2,0).unsqueeze(0)),  # (1,32,32,3)
        ("B,1,H,W,T",    t.permute(1,2,0).unsqueeze(0).unsqueeze(1)),
    ]
    last_err = None
    for name, idxs in candidates:
        try:
            with torch.no_grad():
                recon = decoder.decode(idxs)
            return name, recon
        except Exception as e:
            last_err = e
    raise RuntimeError(f"decode failed for all layouts. last error={repr(last_err)}")

layout_p, past_vid = decode_one(past_tok)
layout_f, fut_vid  = decode_one(fut_tok)
print("decode layouts:", layout_p, layout_f)
print("past recon shape:", tuple(past_vid.shape), past_vid.dtype)
print("fut  recon shape:", tuple(fut_vid.shape), fut_vid.dtype)

# ----------------------------
# 3) (T,H,W,3) uint8 に整形
# ----------------------------
def to_uint8_THWC(x: torch.Tensor):
    x = x.detach().float().cpu()
    if x.ndim != 5:
        raise ValueError(f"decoded tensor must be 5D, got {x.ndim}D")

    # よくある2パターンを吸収
    # (B,3,T,H,W) or (B,T,3,H,W)
    if x.shape[1] == 3:
        x = x[0].permute(1,2,3,0)  # (T,H,W,3)
    elif x.shape[2] == 3:
        x = x[0].permute(0,2,3,1)  # (T,H,W,3)
    else:
        # grayscale (B,1,T,H,W) など
        x = x[0]
        if x.shape[0] == 1:
            x = x[0][..., None].repeat(1,1,3)  # (T,H,W,3)
        else:
            raise ValueError(f"unrecognized channel layout: {tuple(x.shape)}")

    vmin, vmax = float(x.min()), float(x.max())
    # [-1,1] or [0,1] を想定して正規化
    if vmax <= 1.5 and vmin >= -1.5:
        if vmin < 0:
            x = (x + 1.0) / 2.0
        x = x.clamp(0, 1) * 255.0
    else:
        # すでに0..255っぽい場合
        x = x.clamp(0, 255)

    return x.round().to(torch.uint8).numpy(), (vmin, vmax)

past_np, pr = to_uint8_THWC(past_vid)
fut_np,  fr = to_uint8_THWC(fut_vid)
full_np = np.concatenate([past_np, fut_np], axis=0)

print("uint8 video:", full_np.shape, full_np.dtype)
print("past range:", pr, "future range:", fr)

# ----------------------------
# 4) mp4 書き出し & 表示
# ----------------------------
out_gif = Path("/root/work/data/outputs/videos/decoded_sample0.gif")
imageio.mimsave(out_gif, list(full_np), fps=30)
display(out_gif)


decode layouts: B,T,H,W B,T,H,W
past recon shape: (1, 3, 17, 256, 256) torch.bfloat16
fut  recon shape: (1, 3, 17, 256, 256) torch.bfloat16
uint8 video: (34, 256, 256, 3) uint8
past range: (-1.0625, 1.2890625) future range: (-0.9453125, 1.2265625)


PosixPath('/root/work/data/outputs/videos/decoded_sample0.gif')

In [44]:
import os, sys
from pathlib import Path
import numpy as np
import torch
import imageio.v2 as imageio
from tqdm import tqdm

# --- Cosmos Tokenizer import ---
sys.path.append("/root/work/src/Cosmos-Tokenizer/")
from cosmos_tokenizer.video_lib import CausalVideoTokenizer

# ----------------------------
# 設定
# ----------------------------
root = "/root/work/data/raw/train_v2.0"
model_name = "Cosmos-Tokenizer-DV8x8x8"
dec_ckpt = f"/root/work/src/Cosmos-Tokenizer/pretrained_ckpts/{model_name}/decoder.jit"

out_dir = Path("/root/work/data/outputs/videos")
out_dir.mkdir(parents=True, exist_ok=True)


fps = 30                        # GIFのfps
device = "cuda"                 # "cuda" 推奨

# ----------------------------
# Dataset / Decoder
# ----------------------------
# ds = ShardedBlockTokenDataset(
#     root=root,
#     output_format="seq2seq",
#     drop_last_incomplete_block=True,
#     device=None,
# )
ds = ShardedSlidingBlockDataset(
    root=root,
    output_format="seq2seq",
    cache_path=None,  # 一回作れば次から高速
)
decoder = CausalVideoTokenizer(checkpoint_dec=dec_ckpt)

# ----------------------------
# decode helper
# ----------------------------
def decode_one(tok_3x32x32: np.ndarray) -> torch.Tensor:
    """return decoded tensor (5D)"""
    t = torch.from_numpy(tok_3x32x32.astype(np.int64, copy=False)).to(device)

    candidates = [
        t.unsqueeze(0),                              # (1,3,32,32)
        t.unsqueeze(0).unsqueeze(1),                 # (1,1,3,32,32)
        t.permute(1,2,0).unsqueeze(0),               # (1,32,32,3)
        t.permute(1,2,0).unsqueeze(0).unsqueeze(1),  # (1,1,32,32,3)
    ]

    last_err = None
    for idxs in candidates:
        try:
            with torch.no_grad():
                return decoder.decode(idxs)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"decode failed for all layouts. last error={repr(last_err)}")

def to_uint8_THWC(x: torch.Tensor) -> np.ndarray:
    """decoded tensor -> uint8 video (T,H,W,3)"""
    x = x.detach().float().cpu()
    if x.ndim != 5:
        raise ValueError(f"decoded tensor must be 5D, got {x.ndim}D")

    # (B,3,T,H,W) or (B,T,3,H,W)
    if x.shape[1] == 3:
        x = x[0].permute(1,2,3,0)   # (T,H,W,3)
    elif x.shape[2] == 3:
        x = x[0].permute(0,2,3,1)   # (T,H,W,3)
    else:
        # grayscale (B,1,T,H,W) など
        x = x[0]
        if x.shape[0] == 1:
            x = x[0][..., None].repeat(1,1,3)
        else:
            raise ValueError(f"unrecognized channel layout: {tuple(x.shape)}")

    vmin, vmax = float(x.min()), float(x.max())
    # [-1,1] or [0,1] を想定
    if vmax <= 1.5 and vmin >= -1.5:
        if vmin < 0:
            x = (x + 1.0) / 2.0
        x = x.clamp(0, 1) * 255.0
    else:
        x = x.clamp(0, 255)

    return x.round().to(torch.uint8).numpy()

# ----------------------------
# main loop
# ----------------------------
torch.cuda.empty_cache()
start_i, end_i = 0, 50          # inclusive
for i in tqdm(range(start_i, end_i + 1)):
    out_path = out_dir / f"decoded_sample{i}.gif"
    if out_path.exists():
        continue

    sample = ds[i]
    past_tok = sample["past_frames"]     # (3,32,32)
    fut_tok  = sample["future_frames"]  # (3,32,32)

    # decode past/future separately, then concat in time
    past_vid = decode_one(past_tok)
    fut_vid  = decode_one(fut_tok)

    past_np = to_uint8_THWC(past_vid)
    fut_np  = to_uint8_THWC(fut_vid)

    full_np = np.concatenate([past_np, fut_np], axis=0)  # (T,H,W,3)

    # save gif
    imageio.mimsave(out_path, list(full_np), fps=fps)

    # free GPU memory
    del past_vid, fut_vid
    torch.cuda.empty_cache()

print("Done. Saved to:", out_dir)


100%|██████████| 51/51 [01:16<00:00,  1.50s/it]

Done. Saved to: /root/work/data/outputs/videos
